In [ ]:
# | default_exp eval_engine

In [ ]:
# | export
from __future__ import annotations
import json
from collections.abc import Callable
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import structlog
import traceback

# Visualizations
import plotly.express as px
import plotly.graph_objects as go

# Sklearn Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted

log = structlog.get_logger()

In [ ]:
# | export
class FeatureEvaluator:
    """Base class for all feature evaluators.
    Defines the extraction contract that transforms raw DuckDB queries into 1D arrays.

    Subclasses must implement ``extract()`` for per-sample Python extraction.
    Optionally, override ``extract_sql()`` to return a DuckDB SQL query that
    performs full-cohort extraction in a single pass (no Python loop).

    Class Attributes:
        name: Evaluator identifier (used in file names, logging).
        source_file: Parquet suffix (e.g. '.WPS.parquet').
        tier: Feature importance tier (1=core, 2=supplementary).
        category: Feature category for grouping (e.g. 'epigenetics').
        extract_columns: If set, DuckDB will SELECT only these columns
            instead of ``*``. Reduces memory for wide parquet files.
            Must include 'sample_id' (or it will be auto-appended).
        max_chunk_rows: Target rows per DuckDB chunk (default 15M).
            Lower values reduce peak memory for heavy features.
    """

    name: str = "base"
    source_file: str = ".dummy.parquet"
    tier: int = 1
    category: str = "general"
    extract_columns: list[str] | None = None
    max_chunk_rows: int = 15_000_000

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        """Transform the loaded raw dataframe into meaningful scalar metrics.
        Called per sample-group or per sample."""
        raise NotImplementedError(
            "Feature evaluators must implement the extract() method."
        )

    def extract_sql(self) -> str | None:
        """Return a DuckDB SQL query for full-cohort extraction, or None.

        If implemented, the query should:
        - Accept a ``read_parquet(?, ...)`` placeholder for file paths
        - GROUP BY sample_id to produce one row per sample
        - SELECT all extracted feature columns with their final names

        Returning ``None`` (the default) means this evaluator does not
        support SQL pushdown and will use the chunked Python path.
        """
        return None

    @property
    def supports_sql(self) -> bool:
        """True if this evaluator provides a SQL pushdown query."""
        return self.extract_sql() is not None


def parse_array(s) -> list[float]:
    """Parse a numeric array value into a list of floats.

    Handles multiple input formats:
    - numpy arrays (native parquet ``list<float>``, DuckDB ``FLOAT[]``)
    - Python lists/tuples
    - String-encoded arrays (``'[1.0 2.0 3.0]'``) from legacy parquet

    Returns empty list on any parse failure (no silent corruption).
    """
    # Native array types — passthrough (most common path for modern parquets)
    if isinstance(s, np.ndarray):
        return s.tolist()
    if isinstance(s, (list, tuple)):
        try:
            return [float(x) for x in s]
        except (ValueError, TypeError):
            return []

    # String-encoded array — legacy format
    if not isinstance(s, str) or not s.startswith("["):
        return []
    clean = (
        s.replace("[", "").replace("]", "").replace(chr(10), "").replace(chr(13), "")
    )
    try:
        return [float(x) for x in clean.split()]
    except (ValueError, TypeError):
        return []


def univariate_auc(
    feature_col,
    y,
    n_folds: int = 5,
    random_state: int = 42,
) -> float:
    """Compute cross-validated AUC for a single feature using univariate LR.

    Args:
        feature_col: pandas Series or array-like of a single feature.
        y: binary label array (0/1).
        n_folds: number of CV folds.
        random_state: random seed.

    Returns:
        Cross-validated AUC (float). Returns 0.5 if the feature is constant,
        there are too few samples per class, or CV fails.
    """
    import numpy as np

    X = np.array(feature_col).reshape(-1, 1)
    # Replace NaN with 0 for safety
    X = np.nan_to_num(X, nan=0.0)
    y = np.array(y, dtype=int)

    if np.std(X) == 0:
        return 0.5

    try:
        min_class = np.bincount(y).min()
        folds = min(n_folds, min_class)
        if folds < 2:
            return 0.5
        pipe = Pipeline(
            [("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]
        )
        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
        proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
        return float(roc_auc_score(y, proba))
    except Exception as exc:
        log.warning("univariate_auc_failed", error=str(exc))
        return 0.5


def mutual_info_score(
    feature_col,
    y,
    random_state: int = 42,
) -> float:
    """Compute mutual information between a single feature and binary target.

    Uses sklearn's mutual_info_classif with k=3 nearest neighbors to estimate
    the non-linear dependency between a feature and the label. Unlike AUC,
    mutual information captures arbitrary (non-monotonic) relationships.

    Args:
        feature_col: pandas Series or array-like of a single feature.
        y: binary label array (0/1).
        random_state: random seed for reproducibility.

    Returns:
        Mutual information score (float, >= 0). Higher means more informative.
        Returns 0.0 if the feature is constant or computation fails.
    """
    from sklearn.feature_selection import mutual_info_classif

    X = np.array(feature_col).reshape(-1, 1)
    X = np.nan_to_num(X, nan=0.0)
    y = np.array(y, dtype=int)

    if np.std(X) == 0:
        return 0.0

    try:
        mi = mutual_info_classif(X, y, random_state=random_state, n_neighbors=3)
        return float(mi[0])
    except Exception as exc:
        log.warning("mutual_info_failed", error=str(exc))
        return 0.0


In [ ]:
# | export
# Only the 4 ML-relevant tiers. "Insufficient Data" samples are
# excluded from all statistical tests and ML models by design.
LABEL_ORDER = ["True ctDNA+", "Possible ctDNA+", "Possible ctDNA−", "Healthy Normal"]
NEON_COLORS = {
    "True ctDNA+": "#E85D75",  # Pinkish red
    "Possible ctDNA+": "#FF9B42",  # Vivid orange
    "Possible ctDNA−": "gray",
    "Possible ctDNA-": "gray",
    "Healthy Normal": "#8DDA77",  # Bright green
}

CVD_SAFE_COLORS = {
    "True ctDNA+": "#D55E00",  # Vermillion (CVD-safe)
    "Possible ctDNA+": "#E69F00",  # Orange (CVD-safe)
    "Possible ctDNA−": "#999999",  # Gray (CVD-safe)
    "Possible ctDNA-": "#999999",  # Gray fallback
    "Healthy Normal": "#009E73",  # Bluish Green (CVD-safe)
}

LABEL_COLORS = NEON_COLORS.copy()


def set_theme(cvd_safe: bool = False):
    """Dynamically updates the global label and model colors based on CVD preference."""
    global LABEL_COLORS
    LABEL_COLORS.update(CVD_SAFE_COLORS if cvd_safe else NEON_COLORS)


In [ ]:
# | export
def evaluate_feature(
    feature_values: pd.Series,
    labels: pd.Series,
    total_fragments: pd.Series | None = None,
    max_vaf: pd.Series | None = None,
) -> dict:
    """Run all statistical tests for a single feature in one stratum.
    Outputs metrics directly to scoring dict.
    """
    try:
        groups = {}
        for label in LABEL_ORDER:
            mask = labels == label
            # Ensure index alignment if possible
            vals = feature_values[mask].dropna()
            groups[label] = vals

        result = {}

        # --- Kruskal-Wallis (4-group) ---
        non_empty = [g for g in groups.values() if len(g) >= 2]
        if len(non_empty) >= 2:
            try:
                kw_stat, kw_p = stats.kruskal(*non_empty)
                result["kw_statistic"] = kw_stat
                result["kw_pvalue"] = kw_p
            except ValueError as e:
                log.warning("kruskal_failed", error=str(e))

        # --- Pairwise Mann-Whitney U ---
        pairs = [
            ("True ctDNA+", "Healthy Normal"),
            ("Possible ctDNA+", "Healthy Normal"),
            ("True ctDNA+", "Possible ctDNA+"),
            ("Possible ctDNA−", "Healthy Normal"),
            ("True ctDNA+", "Possible ctDNA−"),
        ]
        for g1_name, g2_name in pairs:
            g1, g2 = groups.get(g1_name, []), groups.get(g2_name, [])
            key = (
                f"mwu_{g1_name}_vs_{g2_name}".replace(" ", "_")
                .replace("+", "pos")
                .replace("−", "neg")
            )
            if len(g1) >= 2 and len(g2) >= 2:
                try:
                    u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative="two-sided")
                    r = 1 - (2 * u_stat) / (
                        len(g1) * len(g2)
                    )  # rank-biserial. Note: g1 is always the biologically more "positive" tier (e.g. True+). Positive r means g2 tends to be larger, negative r means g1 tends to be larger.
                    result[f"{key}_pvalue"] = p_val
                    result[f"{key}_effect_size"] = r
                except ValueError as e:
                    log.debug("mwu_failed", pair=f"{g1_name}-{g2_name}", error=str(e))

        # --- FDR correction on pairwise p-values (Benjamini-Hochberg) ---
        pairwise_pval_keys = [
            k for k in result if k.endswith("_pvalue") and k.startswith("mwu_")
        ]
        if len(pairwise_pval_keys) >= 2:
            try:
                from statsmodels.stats.multitest import multipletests

                raw_pvals = [result[k] for k in pairwise_pval_keys]
                _, corrected, _, _ = multipletests(raw_pvals, method="fdr_bh")
                for k, p_corr in zip(pairwise_pval_keys, corrected):
                    result[k.replace("_pvalue", "_pvalue_fdr")] = float(p_corr)
            except ImportError:
                log.warning("statsmodels_not_installed", note="FDR correction skipped")
            except Exception as e:
                log.warning("fdr_correction_failed", error=str(e))

        # --- Cohen's d (True ctDNA+ vs Healthy) ---
        true_pos = groups.get("True ctDNA+", pd.Series())
        healthy = groups.get("Healthy Normal", pd.Series())
        if len(true_pos) >= 2 and len(healthy) >= 2:
            pooled_std = np.sqrt(
                (
                    (len(true_pos) - 1) * true_pos.std() ** 2
                    + (len(healthy) - 1) * healthy.std() ** 2
                )
                / (len(true_pos) + len(healthy) - 2)
            )
            if pooled_std > 0:
                result["cohens_d_true_vs_healthy"] = (
                    true_pos.mean() - healthy.mean()
                ) / pooled_std

        # --- Cohen's d (Possible ctDNA+ vs Healthy) ---
        poss_pos = groups.get("Possible ctDNA+", pd.Series())
        if len(poss_pos) >= 2 and len(healthy) >= 2:
            pooled_std_pp = np.sqrt(
                (
                    (len(poss_pos) - 1) * poss_pos.std() ** 2
                    + (len(healthy) - 1) * healthy.std() ** 2
                )
                / (len(poss_pos) + len(healthy) - 2)
            )
            if pooled_std_pp > 0:
                result["cohens_d_possible_vs_healthy"] = (
                    poss_pos.mean() - healthy.mean()
                ) / pooled_std_pp

        # --- Spearman correlations (Confounders) ---
        if max_vaf is not None:
            valid = feature_values.notna() & max_vaf.notna() & (max_vaf > 0)
            if valid.sum() >= 10:
                try:
                    if (
                        np.var(feature_values[valid]) == 0
                        or np.var(max_vaf[valid]) == 0
                    ):
                        r_val, p_val = 0.0, 1.0
                    else:
                        r_val, p_val = stats.spearmanr(
                            feature_values[valid], max_vaf[valid]
                        )
                    result["spearman_vaf_r"] = r_val
                    result["spearman_vaf_p"] = p_val
                except ValueError as e:
                    log.warning("spearman_vaf_failed", error=str(e))

        if total_fragments is not None:
            valid = feature_values.notna() & total_fragments.notna()
            if valid.sum() >= 10:
                try:
                    if (
                        np.var(feature_values[valid]) == 0
                        or np.var(total_fragments[valid]) == 0
                    ):
                        r_val, p_val = 0.0, 1.0
                    else:
                        r_val, p_val = stats.spearmanr(
                            feature_values[valid], total_fragments[valid]
                        )
                    result["spearman_depth_r"] = r_val
                    result["spearman_depth_p"] = p_val
                except ValueError as e:
                    log.warning("spearman_depth_failed", error=str(e))

        # --- Sample counts ---
        for label, vals in groups.items():
            key = label.replace(" ", "_").replace("+", "pos").replace("−", "neg")
            result[f"n_{key}"] = len(vals)

        result["low_power"] = any(len(v) < 10 for v in groups.values())

        # --- Data quality metrics ---
        result["n_missing"] = int(feature_values.isna().sum())
        result["pct_missing"] = float(feature_values.isna().mean() * 100)
        non_null = feature_values.dropna()
        result["is_zero_variance"] = bool(len(non_null) < 2 or non_null.std() == 0)

        return result
    except Exception as e:
        log.error("evaluate_feature_failed", error=str(e), trace=traceback.format_exc())
        return {"error": str(e)}


In [ ]:
# | export
def plot_violin(
    df: pd.DataFrame, feature_col: str, label_col: str = "label", title: str = ""
) -> go.Figure:
    """4-group violin with overlaid box plot and individual points for small groups."""
    try:
        fig = px.violin(
            df,
            x=label_col,
            y=feature_col,
            color=label_col,
            box=True,
            points="outliers",
            category_orders={label_col: LABEL_ORDER},
            color_discrete_map=LABEL_COLORS,
            title=title,
        )
        fig.update_layout(showlegend=False, xaxis_title="", yaxis_title=feature_col)
        return fig
    except Exception as e:
        log.error("plot_violin_failed", feature=feature_col, error=str(e))
        return go.Figure()


def plot_density(
    df: pd.DataFrame, feature_col: str, label_col: str = "label", title: str = ""
) -> go.Figure:
    """Overlaid density curves per group — shows distribution shape differences."""
    try:
        fig = go.Figure()
        for label in LABEL_ORDER:
            if label not in df[label_col].values:
                continue
            subset = df[df[label_col] == label][feature_col].dropna()
            if len(subset) > 5:
                fig.add_trace(
                    go.Violin(
                        y=subset,
                        name=label,
                        side="positive",
                        line_color=LABEL_COLORS[label],
                        meanline_visible=True,
                        scalemode="width",
                        width=0.8,
                    )
                )
        fig.update_layout(title=title, violinmode="overlay")
        return fig
    except Exception as e:
        log.error("plot_density_failed", feature=feature_col, error=str(e))
        return go.Figure()


def plot_feature_vs_vaf(
    df: pd.DataFrame,
    feature_col: str,
    vaf_col: str = "max_vaf",
    label_col: str = "label",
    title: str = "",
) -> go.Figure:
    """Continuous relationship between feature and tumor burden (VAF proxy)."""
    try:
        cancer = df[df[vaf_col] > 0]  # exclude healthy & zero-VAF
        if cancer.empty:
            return go.Figure()
        fig = px.scatter(
            cancer,
            x=vaf_col,
            y=feature_col,
            color=label_col,
            color_discrete_map=LABEL_COLORS,
            opacity=0.5,
            trendline="lowess",
            log_x=True,
            title=title,
        )
        return fig
    except Exception as e:
        log.error("plot_vs_vaf_failed", feature=feature_col, error=str(e))
        return go.Figure()


def plot_roc_curves(
    y_true_dict: dict[str, np.ndarray],
    y_score_dict: dict[str, np.ndarray],
    title: str = "",
) -> go.Figure:
    """Overlay ROC curves for multiple comparisons."""
    try:
        fig = go.Figure()
        for name, y_true in y_true_dict.items():
            if name not in y_score_dict:
                continue
            y_score = y_score_dict[name]
            if len(np.unique(y_true)) < 2:
                continue  # Cannot plot ROC with 1 class
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            fig.add_trace(
                go.Scatter(
                    x=fpr,
                    y=tpr,
                    name=f"{name} (AUC={roc_auc:.3f})",
                    mode="lines",
                )
            )
        fig.add_trace(
            go.Scatter(
                x=[0, 1],
                y=[0, 1],
                mode="lines",
                line=dict(dash="dash", color="gray"),
                name="Random",
            )
        )
        fig.update_layout(
            title=title,
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate",
        )
        return fig
    except Exception as e:
        log.error("plot_roc_curves_failed", error=str(e))
        return go.Figure()


def plot_feature_importance(
    importances: dict[str, float], title: str = ""
) -> go.Figure:
    """Bar plot of RF feature importances."""
    try:
        df = pd.DataFrame(
            {
                "feature": list(importances.keys()),
                "importance": list(importances.values()),
            }
        )
        df = df.sort_values("importance", ascending=True).tail(20)  # Top 20
        fig = px.bar(df, x="importance", y="feature", orientation="h", title=title)
        return fig
    except Exception as e:
        log.error("plot_rf_importance_failed", error=str(e))
        return go.Figure()


def decision_curve_analysis(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresholds: np.ndarray | None = None,
) -> dict:
    """Compute Decision Curve Analysis (DCA) net benefit data.

    For each threshold, calculates the net benefit of using the model vs
    treating all or treating none. This helps clinicians choose an operating
    threshold that balances false positives against missed detections.

    Args:
        y_true: Binary ground truth labels (0/1).
        y_prob: Predicted probabilities for positive class.
        thresholds: Array of decision thresholds to evaluate.
            Defaults to ``np.linspace(0.01, 0.99, 99)``.

    Returns:
        Dictionary with keys ``thresholds``, ``net_benefit_model``, and
        ``net_benefit_treat_all``.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    n = len(y_true)
    prevalence = float(y_true.mean())
    net_benefits = []
    treat_all = []

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        odds = t / (1 - t) if t < 1 else float("inf")
        nb = (tp / n) - (fp / n) * odds
        net_benefits.append(float(nb))
        ta = prevalence - (1 - prevalence) * odds
        treat_all.append(float(ta))

    return {
        "thresholds": thresholds.tolist(),
        "net_benefit_model": net_benefits,
        "net_benefit_treat_all": treat_all,
    }


In [ ]:
# | export
# ── Shared Metric Helpers ─────────────────────────────────────────────────────
# Extracted from single_feature_model() for reuse by evaluate_model(),
# cpu_models(), and gpu_models(). No closures — pure functions.


def _optimal_threshold(y_true: np.ndarray, y_pred_prob: np.ndarray) -> float:
    """Find threshold maximizing Youden's J (sensitivity + specificity - 1).

    Falls back to 0.5 if roc_curve fails (e.g. single-class fold).
    """
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
        if len(thresholds) > 0:
            optimal_idx = np.argmax(tpr - fpr)
            return float(thresholds[optimal_idx])
    except Exception:
        log.debug("optimal_threshold_fallback", reason="roc_curve failed")
    return 0.5


def _classification_metrics(
    y_true: np.ndarray, y_pred_prob: np.ndarray, threshold: float = 0.5
) -> tuple[dict, list]:
    """Compute classification report and confusion matrix at given threshold.

    Returns (classification_report_dict, confusion_matrix_list).
    """
    from sklearn.metrics import classification_report, confusion_matrix

    y_pred = (y_pred_prob >= threshold).astype(int)
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred).tolist()
    return rep, cm


def _bootstrap_auc(
    y_true: np.ndarray,
    y_score: np.ndarray,
    n_boot: int = 1000,
    seed: int = 42,
) -> tuple[float | None, float | None]:
    """Compute 95% bootstrap CI for AUC-ROC.

    Returns (lower, upper) or (None, None) if fewer than 10 valid
    bootstrap samples could be drawn (e.g. very small dataset).
    """
    rng = np.random.RandomState(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    if len(aucs) < 10:
        return None, None
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def _per_fold_auc(
    y: np.ndarray,
    oof_probs: np.ndarray,
    cv: StratifiedKFold,
    X: np.ndarray,
) -> list[float]:
    """Compute per-fold AUC from out-of-fold predictions.

    Uses the same CV split that produced the OOF predictions
    (D-01 fix: no independent cross_val_score run).
    Skips folds with a single class.
    """
    fold_aucs = []
    for _, test_idx in cv.split(X, y):
        if len(np.unique(y[test_idx])) < 2:
            continue
        fold_aucs.append(float(roc_auc_score(y[test_idx], oof_probs[test_idx])))
    return fold_aucs


def _pr_curve(
    y_true: np.ndarray, y_prob: np.ndarray
) -> tuple[dict | None, float | None]:
    """Compute precision-recall curve and average precision.

    Returns (pr_dict, avg_precision) or (None, None) on failure.
    """
    try:
        from sklearn.metrics import precision_recall_curve, average_precision_score

        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        pr_dict = {"precision": prec.tolist(), "recall": rec.tolist()}
        avg_prec = float(average_precision_score(y_true, y_prob))
        return pr_dict, avg_prec
    except Exception as e:
        log.warning("pr_curve_failed", error=str(e))
        return None, None


def _calibration(y_true: np.ndarray, y_prob: np.ndarray) -> dict | None:
    """Compute calibration curve (10 uniform bins).

    Returns dict with prob_true/prob_pred or None on failure.
    """
    try:
        from sklearn.calibration import calibration_curve

        prob_true, prob_pred = calibration_curve(
            y_true, y_prob, n_bins=10, strategy="uniform"
        )
        return {"prob_true": prob_true.tolist(), "prob_pred": prob_pred.tolist()}
    except Exception as e:
        log.warning("calibration_failed", error=str(e))
        return None


def _subgroup_metrics(
    mask: np.ndarray,
    y_all: np.ndarray,
    preds_dict: dict[str, np.ndarray],
) -> dict:
    """Compute sensitivity/specificity for each model on a subgroup.

    Args:
        mask: boolean array selecting the subgroup.
        y_all: full label array.
        preds_dict: {model_name: binary_predictions_array}.

    Returns dict with {model}_sensitivity and {model}_specificity keys.
    """
    from sklearn.metrics import confusion_matrix

    res = {}
    y_sub = y_all[mask]
    if len(y_sub) < 5 or len(np.unique(y_sub)) < 2:
        return res

    for model_name, preds in preds_dict.items():
        if preds is None:
            continue
        try:
            p_sub = preds[mask]
            tn, fp, fn, tp = confusion_matrix(y_sub, p_sub, labels=[0, 1]).ravel()
            res[f"{model_name}_sensitivity"] = (
                float(tp / (tp + fn)) if (tp + fn) > 0 else None
            )
            res[f"{model_name}_specificity"] = (
                float(tn / (tn + fp)) if (tn + fp) > 0 else None
            )
        except Exception as e:
            log.warning("subgroup_metrics_failed", model=model_name, error=str(e))

    return res


# ── Feature Group Ablation Utilities ──────────────────────────────────────────

# Suffix → group name mapping for identify_feature_groups().
# Order matters: longer suffixes checked first to avoid partial matches.
# Source: all evaluator extract() methods in kreview/features/*.py
_SUFFIX_TO_GROUP: list[tuple[str, str]] = [
    # FSC gene ratios (check before generic _ratio)
    ("_ultra_short_ratio", "ultra_short_ratio"),
    ("_core_short_ratio", "core_short_ratio"),
    ("_mono_nucl_ratio", "mono_nucl_ratio"),
    ("_di_nucl_ratio", "di_nucl_ratio"),
    ("_long_ratio", "long_ratio"),
    # FSC gene depth
    ("_normalized_depth", "normalized_depth"),
    ("_normalized_depth_cv", "normalized_depth_cv"),
    # TFBS / ATAC stats
    ("_mean_size", "mean_size"),
    ("_z_score", "z_score"),
    # WPS / FragProfile stats (multi-word suffixes first)
    ("_spectral_dominant_freq", "spectral_dominant_freq"),
    ("_peak_valley", "peak_valley"),
    ("_143_166_ratio", "143_166_ratio"),
    ("_bimodality_index", "bimodality_index"),
    ("_top10_concentration", "top10_concentration"),
    ("_purine_pyrimidine_ratio", "purine_pyrimidine_ratio"),
    ("_dnase1l3_score", "dnase1l3_score"),
    ("_shannon_entropy", "shannon_entropy"),
    ("_variance_across_genes", "variance_across_genes"),
    ("_fsr_gw_median", "fsr_gw_median"),
    ("_fsr_gw_std", "fsr_gw_std"),
    # OCF
    ("_OCF", "OCF"),
    ("_ocf_z", "ocf_z"),
    # Generic single-word suffixes (check last)
    ("_entropy", "entropy"),
    ("_count", "count"),
    ("_mean", "mean"),
    ("_median", "median"),
    ("_std", "std"),
    ("_cv", "cv"),
    ("_mad", "mad"),
]


def identify_feature_groups(
    columns: list[str],
    meta_cols: set[str] | None = None,
) -> dict[str, list[str]]:
    """Map selected feature column names → feature groups by statistic type.

    Groups are identified by suffix pattern matching against known kreview
    evaluator output patterns (see ``_SUFFIX_TO_GROUP``).  Columns that don't
    match any known suffix go into the ``"other"`` group.

    Parameters
    ----------
    columns : list[str]
        Feature column names from a selected matrix.
    meta_cols : set[str] | None
        Columns to exclude (e.g. ``LABEL_META_COLS``).  If *None*, uses
        ``kreview.core.LABEL_META_COLS``.

    Returns
    -------
    dict[str, list[str]]
        Mapping of group name → list of column names belonging to that group.
    """
    if meta_cols is None:
        from kreview.core import LABEL_META_COLS

        meta_cols = LABEL_META_COLS

    groups: dict[str, list[str]] = {}
    for col in columns:
        if col in meta_cols:
            continue
        matched = False
        for suffix, group_name in _SUFFIX_TO_GROUP:
            if col.endswith(suffix):
                groups.setdefault(group_name, []).append(col)
                matched = True
                break
        if not matched:
            # Columns starting with special prefixes get their own groups
            if col.startswith("bpmotif_"):
                groups.setdefault("breakpoint_motif", []).append(col)
            elif col.startswith("mds_gw_"):
                groups.setdefault("mds_genomewide", []).append(col)
            elif col.startswith("global_"):
                groups.setdefault("global_aggregate", []).append(col)
            else:
                groups.setdefault("other", []).append(col)

    log.info(
        "feature_groups_identified",
        n_groups=len(groups),
        group_sizes={k: len(v) for k, v in groups.items()},
        n_columns=len(columns),
        n_excluded_meta=sum(1 for c in columns if c in meta_cols),
    )
    return groups


def generate_subsets(
    groups: dict[str, list[str]],
) -> dict[str, list[str]]:
    """Generate feature subsets for ablation from feature groups.

    Returns a dict of named subsets:

    - ``"ALL"``: all features from all groups
    - ``"{group}"``: solo — only features from that group (one per group)
    - ``"no_{group}"``: leave-one-out — everything except that group

    If only 1 group exists, returns only ``{"ALL": all_features}`` since
    ablation is meaningless with a single group.

    Parameters
    ----------
    groups : dict[str, list[str]]
        Output of :func:`identify_feature_groups`.

    Returns
    -------
    dict[str, list[str]]
        Named subsets mapping subset name → list of feature column names.
    """
    all_features: list[str] = []
    for cols in groups.values():
        all_features.extend(cols)

    subsets = {"ALL": sorted(all_features)}

    if len(groups) <= 1:
        log.info("single_feature_group_skip_ablation", n_features=len(all_features))
        return subsets

    for group_name, group_cols in groups.items():
        # Solo: just this group
        subsets[group_name] = sorted(group_cols)
        # Leave-one-out: everything except this group
        remaining = [c for c in all_features if c not in set(group_cols)]
        subsets[f"no_{group_name}"] = sorted(remaining)

    log.info(
        "ablation_subsets_generated",
        n_groups=len(groups),
        n_subsets=len(subsets),
        subset_sizes={k: len(v) for k, v in subsets.items()},
    )
    return subsets


def _inner_cv_sensitivity(
    model,
    X_train: np.ndarray,
    y_train: np.ndarray,
    sample_labels_train: np.ndarray,
    n_inner_folds: int = 3,
    random_state: int = 42,
) -> float:
    """Lightweight inner-CV sensitivity@100%spec_healthy for ablation.

    Runs a stratified k-fold on the *training* portion of a single outer
    fold.  Only computes the one metric used to rank subsets — no
    classification report, SHAP, ROC curves, etc.

    ~10× faster than :func:`evaluate_model` because it skips all
    diagnostic outputs.

    Parameters
    ----------
    model
        Unfitted sklearn estimator (will be cloned per fold).
    X_train : np.ndarray
        Feature matrix for the training split of an outer fold.
    y_train : np.ndarray
        Binary labels (0/1) for the training split.
    sample_labels_train : np.ndarray
        Original 4-tier labels (e.g. ``"Healthy Normal"``) aligned
        with ``X_train``, needed to identify healthy normals.
    n_inner_folds : int
        Number of inner CV folds (default 3).
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    float
        Mean sensitivity@100%spec_healthy across inner folds.
        Returns 0.0 if computation fails (e.g. too few samples).
    """
    from sklearn.base import clone

    if len(np.unique(y_train)) < 2:
        log.debug("inner_cv_single_class", n_samples=len(y_train))
        return 0.0

    # Need at least n_inner_folds samples per class
    n_pos = int(y_train.sum())
    n_neg = len(y_train) - n_pos
    if n_pos < n_inner_folds or n_neg < n_inner_folds:
        log.debug(
            "inner_cv_insufficient_samples",
            n_pos=n_pos,
            n_neg=n_neg,
            n_inner_folds=n_inner_folds,
        )
        return 0.0

    inner_cv = StratifiedKFold(
        n_splits=n_inner_folds, shuffle=True, random_state=random_state
    )

    fold_sensitivities = []
    for fold_idx, (tr_idx, val_idx) in enumerate(inner_cv.split(X_train, y_train)):
        try:
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]
            labels_val = sample_labels_train[val_idx]

            m = clone(model)
            m.fit(X_tr, y_tr)

            if hasattr(m, "predict_proba"):
                probs = m.predict_proba(X_val)[:, 1]
            elif hasattr(m, "decision_function"):
                probs = m.decision_function(X_val)
            else:
                log.warning("inner_cv_no_predict_proba", fold=fold_idx)
                fold_sensitivities.append(0.0)
                continue

            # Compute sensitivity@100%spec_healthy
            healthy_mask = np.array([lbl == "Healthy Normal" for lbl in labels_val])
            if not healthy_mask.any():
                # No healthy normals in this validation fold — skip
                continue

            max_healthy_prob = float(probs[healthy_mask].max())
            pos_mask = y_val == 1
            n_pos_val = int(pos_mask.sum())
            if n_pos_val > 0:
                detected = int((probs[pos_mask] > max_healthy_prob).sum())
                fold_sensitivities.append(detected / n_pos_val)
            else:
                fold_sensitivities.append(0.0)

        except Exception as e:
            log.debug("inner_cv_fold_error", fold=fold_idx, error=str(e))
            fold_sensitivities.append(0.0)

    if not fold_sensitivities:
        return 0.0

    return float(np.mean(fold_sensitivities))


def ablate_feature_groups(
    matrix_path: Path | str,
    models: tuple[str, ...] = ("lr", "rf", "xgb"),
    n_outer_folds: int = 5,
    n_inner_folds: int = 3,
    random_state: int = 42,
    output_path: Path | str | None = None,
    device: str = "cpu",
    max_gpu_features: int | None = None,
    eval_stats_path: Path | str | None = None,
) -> dict:
    """Run feature group ablation with per-fold nested cross-validation.

    For each outer fold, evaluates all feature subsets using inner CV
    to find the best subset per model.  Results include per-fold winners
    and the fold assignment array for downstream reproducibility.

    **Data flow**:

    1. Load matrix, filter to ``split == "train"`` (prevent holdout leakage).
    2. Build binary target via :func:`kreview.selection.build_binary_target`.
    3. Identify feature groups via :func:`identify_feature_groups`.
    4. Generate subsets via :func:`generate_subsets`.
    5. For each outer fold:
       a. Split into outer-train / outer-val using StratifiedKFold.
       b. For each model × subset:
          - Run :func:`_inner_cv_sensitivity` on outer-train.
          - Record sensitivity@100%spec_healthy score.
       c. Identify winning subset per model for this fold.
    6. Aggregate results across folds.

    Parameters
    ----------
    matrix_path : Path | str
        Path to ``*_matrix.parquet`` file.
    models : tuple[str, ...]
        Model names to evaluate. CPU: ``"lr"``, ``"rf"``, ``"xgb"``.
        GPU: ``"tabpfn"``, ``"tabicl"`` (zero-shot only).
    n_outer_folds : int
        Number of outer CV folds (must match downstream EVAL step).
    n_inner_folds : int
        Number of inner CV folds for subset selection.
    random_state : int
        Random seed for reproducibility.
    output_path : Path | str | None
        Directory to write ablation JSON results.
    device : str
        PyTorch device for GPU models (default ``"cpu"``).
    max_gpu_features : int | None
        Feature cap for GPU models (TabPFN limit).
    eval_stats_path : Path | str | None
        Path to ``*_eval_stats.parquet`` for score-based feature capping.

    Returns
    -------
    dict
        Ablation results containing per-fold per-model scores, winners,
        fold assignments, and feature group metadata.
    """
    from kreview.core import LABEL_META_COLS
    from kreview.selection import build_binary_target, _impute

    matrix_path = Path(matrix_path)
    evaluator = matrix_path.stem.replace("_matrix", "")

    log.info(
        "ablation_start",
        evaluator=evaluator,
        models=models,
        n_outer=n_outer_folds,
        n_inner=n_inner_folds,
    )

    # ── 1. Load matrix and filter to train split ──
    df = pd.read_parquet(matrix_path)
    log.info("ablation_loaded", shape=df.shape, evaluator=evaluator)

    if "split" in df.columns:
        train_mask = df["split"] == "train"
        n_excluded = int((~train_mask).sum())
        df = df[train_mask].copy()
        log.info("ablation_train_filter", n_train=len(df), n_holdout=n_excluded)
    else:
        log.warning(
            "ablation_no_split_column",
            evaluator=evaluator,
            impact="using all samples — holdout leakage risk",
        )

    # ── 2. Build binary target ──
    model_df, y = build_binary_target(df, label_col="label")
    sample_labels = model_df["label"].values  # 4-tier labels for inner CV

    # ── 3. Identify feature columns and groups ──
    feature_cols = [
        c
        for c in model_df.select_dtypes(include=np.number).columns
        if c not in LABEL_META_COLS
    ]
    if not feature_cols:
        log.error("ablation_no_features", evaluator=evaluator)
        return {"evaluator": evaluator, "error": "no_feature_columns"}

    # Impute NaNs and drop zero-variance
    model_df[feature_cols] = _impute(model_df[feature_cols], "median")
    nonconst = [c for c in feature_cols if model_df[c].std() > 0]
    if len(nonconst) < len(feature_cols):
        log.info(
            "ablation_dropped_zero_var", n_dropped=len(feature_cols) - len(nonconst)
        )
    feature_cols = nonconst

    groups = identify_feature_groups(feature_cols)
    subsets = generate_subsets(groups)

    if len(subsets) == 1:
        # Single group → passthrough, no ablation needed
        log.info(
            "ablation_passthrough", evaluator=evaluator, reason="single_feature_group"
        )
        result = {
            "evaluator": evaluator,
            "passthrough": True,
            "n_features": len(feature_cols),
            "n_groups": 1,
            "groups": {k: len(v) for k, v in groups.items()},
        }
        if output_path:
            _save_ablation_json(result, output_path, evaluator, models)
        return result

    # ── 4. Build model factories ──
    model_factories = _build_ablation_model_factories(models, device)

    # ── 5. Optional feature capping for GPU models ──
    capped_features = None
    if max_gpu_features and eval_stats_path:
        capped_features = _load_feature_cap(
            eval_stats_path, feature_cols, max_gpu_features
        )

    # ── 6. Outer CV loop ──
    outer_cv = StratifiedKFold(
        n_splits=n_outer_folds, shuffle=True, random_state=random_state
    )
    fold_assignment = np.full(len(y), -1, dtype=int)

    per_fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(
        outer_cv.split(model_df[feature_cols].values, y)
    ):
        fold_assignment[val_idx] = fold_idx
        fold_result = {"fold": fold_idx, "models": {}}

        X_full = model_df[feature_cols].values
        y_full = y

        X_outer_train = X_full[train_idx]
        y_outer_train = y_full[train_idx]
        labels_outer_train = sample_labels[train_idx]

        log.info(
            "ablation_outer_fold",
            fold=fold_idx,
            n_train=len(train_idx),
            n_val=len(val_idx),
        )

        for model_name, model_factory in model_factories.items():
            model_scores: dict[str, float] = {}

            for subset_name, subset_cols in subsets.items():
                # Map subset columns to indices in feature_cols
                col_indices = [
                    feature_cols.index(c) for c in subset_cols if c in feature_cols
                ]
                if not col_indices:
                    model_scores[subset_name] = 0.0
                    continue

                # Apply GPU feature cap if needed
                if (
                    max_gpu_features
                    and model_name in _GPU_MODEL_NAMES
                    and len(col_indices) > max_gpu_features
                ):
                    if capped_features:
                        # Score-based: keep only top features by univariate AUC
                        cap_set = set(capped_features[:max_gpu_features])
                        col_indices = [
                            i for i in col_indices if feature_cols[i] in cap_set
                        ]
                    else:
                        # Variance-based fallback: keep highest-variance features
                        variances = np.var(X_outer_train[:, col_indices], axis=0)
                        top_k = np.argsort(variances)[-max_gpu_features:]
                        col_indices = [col_indices[i] for i in top_k]

                X_subset = X_outer_train[:, col_indices]

                try:
                    model_instance = model_factory()
                    if model_instance is None:
                        # GPU model unavailable (e.g., tabpfn not installed)
                        # — skip silently, already warned in factory
                        model_scores[subset_name] = 0.0
                        continue
                    score = _inner_cv_sensitivity(
                        model_instance,
                        X_subset,
                        y_outer_train,
                        labels_outer_train,
                        n_inner_folds=n_inner_folds,
                        random_state=random_state + fold_idx,
                    )
                    model_scores[subset_name] = score
                except Exception as e:
                    log.warning(
                        "ablation_inner_cv_error",
                        model=model_name,
                        subset=subset_name,
                        fold=fold_idx,
                        error=str(e),
                    )
                    model_scores[subset_name] = 0.0

            # Determine winner for this model in this fold
            if model_scores:
                winner = max(model_scores, key=model_scores.get)
            else:
                winner = "ALL"

            fold_result["models"][model_name] = {  # type: ignore[index]
                "scores": model_scores,
                "winner": winner,
                "winner_score": model_scores.get(winner, 0.0),
                "winner_features": subsets.get(winner, subsets["ALL"]),
            }

            log.info(
                "ablation_fold_model_done",
                fold=fold_idx,
                model=model_name,
                winner=winner,
                winner_score=f"{model_scores.get(winner, 0.0):.4f}",
                n_subsets_evaluated=len(model_scores),
            )

        per_fold_results.append(fold_result)

    # ── 7. Build result dict ──
    result = {
        "evaluator": evaluator,
        "passthrough": False,
        "n_outer_folds": n_outer_folds,
        "n_inner_folds": n_inner_folds,
        "n_features_total": len(feature_cols),
        "n_groups": len(groups),
        "groups": {k: len(v) for k, v in groups.items()},
        "group_details": {k: sorted(v) for k, v in groups.items()},
        "subsets": {k: len(v) for k, v in subsets.items()},
        "fold_assignment": fold_assignment.tolist(),
        "per_fold_results": per_fold_results,
        "models_evaluated": list(model_factories.keys()),
        "random_state": random_state,
    }

    log.info(
        "ablation_complete",
        evaluator=evaluator,
        n_folds=n_outer_folds,
        n_models=len(model_factories),
    )

    # ── 8. Save output ──
    if output_path:
        _save_ablation_json(result, output_path, evaluator, models)

    return result


def _build_ablation_model_factories(
    models: tuple[str, ...],
    device: str = "cpu",
) -> dict[str, Callable]:
    """Create model factory functions for ablation.

    Returns a dict mapping model name → zero-argument callable that returns
    an unfitted estimator instance.

    Parameters
    ----------
    models : tuple[str, ...]
        Model names (e.g. ``("lr", "rf", "xgb")``).
    device : str
        PyTorch device for GPU models.

    Returns
    -------
    dict[str, Callable]
        Model name → factory function.
    """
    factories: dict[str, Callable] = {}

    for name in models:
        if name == "lr":
            factories["lr"] = lambda: Pipeline(
                [
                    ("scaler", StandardScaler()),
                    (
                        "clf",
                        LogisticRegression(
                            max_iter=2000,
                            solver="lbfgs",
                            class_weight="balanced",
                            random_state=42,
                        ),
                    ),
                ]
            )
        elif name == "rf":
            factories["rf"] = lambda: RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )
        elif name == "xgb":

            def _xgb_factory():
                try:
                    from xgboost import XGBClassifier

                    return XGBClassifier(
                        n_estimators=300,
                        max_depth=6,
                        learning_rate=0.1,
                        scale_pos_weight=1.0,
                        use_label_encoder=False,
                        eval_metric="logloss",
                        random_state=42,
                        n_jobs=-1,
                        verbosity=0,
                    )
                except ImportError:
                    log.warning("xgb_not_available", fallback="rf")
                    return RandomForestClassifier(
                        n_estimators=500,
                        max_depth=8,
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1,
                    )

            factories["xgb"] = _xgb_factory
        elif name in ("tabpfn", "tabicl"):
            # Zero-shot only for inner CV ablation — use the shared builder
            # which correctly wraps the model in GPUModelCVAdapter.
            _name = name
            _device = device

            def _gpu_factory(n=_name, d=_device):
                model = _build_gpu_model(name=n, device=d)
                if model is None:
                    log.warning(
                        "ablation_gpu_model_unavailable",
                        model=n,
                        hint="GPU library not installed — ablation will score 0.0",
                    )
                return model

            factories[name] = _gpu_factory
        else:
            log.warning("unknown_ablation_model", name=name)

    return factories


def _load_feature_cap(
    eval_stats_path: Path | str,
    feature_cols: list[str],
    max_features: int,
) -> list[str] | None:
    """Load eval_stats and return top features by univariate AUC for capping.

    Parameters
    ----------
    eval_stats_path : Path | str
        Path to ``*_eval_stats.parquet``.
    feature_cols : list[str]
        Available feature columns.
    max_features : int
        Maximum number of features to return.

    Returns
    -------
    list[str] | None
        Ordered list of top features, or None if loading fails.
    """
    try:
        stats_df = pd.read_parquet(eval_stats_path)
        if (
            "feature_column" not in stats_df.columns
            or "univariate_auc" not in stats_df.columns
        ):
            log.warning("eval_stats_missing_columns", path=str(eval_stats_path))
            return None
        available = stats_df[stats_df["feature_column"].isin(feature_cols)]
        top = available.nlargest(max_features, "univariate_auc")[
            "feature_column"
        ].tolist()
        log.info("feature_cap_loaded", n_capped=len(top), max_features=max_features)
        return top
    except Exception as e:
        log.warning("eval_stats_load_failed", error=str(e))
        return None


def _save_ablation_json(
    result: dict,
    output_path: Path | str,
    evaluator: str,
    models: tuple[str, ...],
) -> Path:
    """Save ablation results to a JSON file.

    File naming convention: ``{evaluator}_ablation_{type}.json`` where
    type is ``cpu`` or ``gpu`` based on the model names.

    Parameters
    ----------
    result : dict
        Ablation results dict.
    output_path : Path | str
        Output directory.
    evaluator : str
        Evaluator name (e.g. ``"TfbsOnTarget"``).
    models : tuple[str, ...]
        Model names used (determines cpu/gpu suffix).

    Returns
    -------
    Path
        Path to the saved JSON file.
    """
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    is_gpu = any(m in _GPU_MODEL_NAMES for m in models)
    suffix = "gpu" if is_gpu else "cpu"
    filename = f"{evaluator}_ablation_{suffix}.json"
    out_path = output_path / filename

    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, default=str)

    log.info("ablation_saved", path=str(out_path), size_bytes=out_path.stat().st_size)
    return out_path


def merge_ablation(
    cpu_json_path: Path | str,
    gpu_json_path: Path | str | None = None,
    matrix_path: Path | str | None = None,
    output_path: Path | str | None = None,
) -> dict:
    """Merge CPU and GPU ablation results into a unified best_subset.json.

    For each model across all folds, determines the per-fold winning
    feature subset.  The output JSON is consumed by downstream
    ``kreview eval cpu`` / ``kreview eval gpu`` via ``--best-subset``.

    Handles GPU error JSONs gracefully: if the GPU JSON contains an
    ``"error"`` key (from a failed GPU ablation), the merge proceeds
    with CPU-only results.

    Parameters
    ----------
    cpu_json_path : Path | str
        Path to ``*_ablation_cpu.json`` from :func:`ablate_feature_groups`.
    gpu_json_path : Path | str | None
        Path to ``*_ablation_gpu.json``.  ``None`` or a path to a
        file containing ``"error"`` triggers CPU-only mode.
    matrix_path : Path | str | None
        Path to the original matrix parquet (for fold assignment
        verification).
    output_path : Path | str | None
        Directory to write ``{evaluator}_best_subset.json``.

    Returns
    -------
    dict
        Merged result with ``per_model_per_fold_features`` and
        ``fold_assignment``.
    """
    cpu_json_path = Path(cpu_json_path)
    with open(cpu_json_path) as f:
        cpu_result = json.load(f)

    evaluator = cpu_result.get("evaluator", "unknown")

    # Handle passthrough (single-group evaluators)
    if cpu_result.get("passthrough", False):
        log.info("merge_ablation_passthrough", evaluator=evaluator)
        merged = {
            "evaluator": evaluator,
            "passthrough": True,
            "per_model_per_fold_features": {},
            "fold_assignment": cpu_result.get("fold_assignment", []),
        }
        if output_path:
            _save_merged_json(merged, output_path, evaluator)
        return merged

    # Load GPU results (optional, may have errors)
    gpu_result = None
    gpu_error = False
    if gpu_json_path:
        gpu_json_path = Path(gpu_json_path)
        if gpu_json_path.exists() and gpu_json_path.name != "NO_GPU_ABLATION":
            try:
                with open(gpu_json_path) as f:
                    gpu_result = json.load(f)
                if "error" in gpu_result:
                    log.warning(
                        "merge_ablation_gpu_error",
                        evaluator=evaluator,
                        error=gpu_result["error"],
                    )
                    gpu_error = True
                    gpu_result = None
            except Exception as e:
                log.warning(
                    "merge_ablation_gpu_load_failed", evaluator=evaluator, error=str(e)
                )
                gpu_error = True

    # Combine per-fold results from CPU and GPU
    n_folds = cpu_result.get("n_outer_folds", 5)
    cpu_folds = cpu_result.get("per_fold_results", [])
    gpu_folds = gpu_result.get("per_fold_results", []) if gpu_result else []

    per_model_per_fold: dict[str, dict] = {}

    for fold_idx in range(n_folds):
        # CPU results for this fold
        if fold_idx < len(cpu_folds):
            for model_name, model_data in cpu_folds[fold_idx].get("models", {}).items():
                if model_name not in per_model_per_fold:
                    per_model_per_fold[model_name] = {"folds": {}}
                per_model_per_fold[model_name]["folds"][str(fold_idx)] = {
                    "winner": model_data["winner"],
                    "winner_score": model_data["winner_score"],
                    "features": model_data["winner_features"],
                }

        # GPU results for this fold
        if not gpu_error and fold_idx < len(gpu_folds):
            for model_name, model_data in gpu_folds[fold_idx].get("models", {}).items():
                if model_name not in per_model_per_fold:
                    per_model_per_fold[model_name] = {"folds": {}}
                per_model_per_fold[model_name]["folds"][str(fold_idx)] = {
                    "winner": model_data["winner"],
                    "winner_score": model_data["winner_score"],
                    "features": model_data["winner_features"],
                }

    merged = {
        "evaluator": evaluator,
        "passthrough": False,
        "n_outer_folds": n_folds,
        "fold_assignment": cpu_result.get("fold_assignment", []),
        "per_model_per_fold_features": per_model_per_fold,
        "groups": cpu_result.get("groups", {}),
        "group_details": cpu_result.get("group_details", {}),
        "gpu_error": gpu_error,
        "models_merged": list(per_model_per_fold.keys()),
    }

    log.info(
        "merge_ablation_complete",
        evaluator=evaluator,
        n_models=len(per_model_per_fold),
        n_folds=n_folds,
        gpu_error=gpu_error,
    )

    if output_path:
        _save_merged_json(merged, output_path, evaluator)

    return merged


def _save_merged_json(
    result: dict,
    output_path: Path | str,
    evaluator: str,
) -> Path:
    """Save merged ablation result as best_subset.json."""
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / f"{evaluator}_best_subset.json"
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2, default=str)
    log.info(
        "best_subset_saved", path=str(out_path), size_bytes=out_path.stat().st_size
    )
    return out_path


def _compute_oof_metrics(
    y: np.ndarray,
    oof_probs: np.ndarray,
    name: str,
    feature_names: list[str] | None = None,
    sample_labels: np.ndarray | None = None,
    random_state: int = 42,
    fold_assignment: np.ndarray | None = None,
) -> dict:
    """Compute all standard metrics from stitched OOF probabilities.

    Shared between the standard path (``cross_val_predict``) and the
    nested CV path (manual fold iteration with per-fold features).
    Produces the same keys that :func:`evaluate_model` would.

    Parameters
    ----------
    y : np.ndarray
        Binary labels (0/1).
    oof_probs : np.ndarray
        Out-of-fold predicted probabilities, one per sample.
    name : str
        Model name prefix (e.g. ``"lr"``, ``"rf"``).
    feature_names : list[str] | None
        Feature column names (for logging only).
    sample_labels : np.ndarray | None
        Original 4-tier labels for healthy-normal specificity.
    random_state : int
        Random seed for bootstrap CI.
    fold_assignment : np.ndarray | None
        Per-sample fold indices for per-fold AUC computation.

    Returns
    -------
    dict
        Standard metric keys including ``auc_{name}``, sensitivity at
        fixed specificity, healthy-normal threshold, optimal threshold,
        classification report, confusion matrix, fold AUCs, etc.
    """
    result = {}

    # AUC
    try:
        result[f"auc_{name}"] = float(roc_auc_score(y, oof_probs))
    except ValueError:
        result[f"auc_{name}"] = 0.5
    result[f"{name}_oof_probs"] = np.round(oof_probs, 6).tolist()

    # Bootstrap CI
    try:
        ci_lo, ci_hi = _bootstrap_auc(y, oof_probs, seed=random_state)
        result[f"auc_{name}_ci_lower"] = ci_lo
        result[f"auc_{name}_ci_upper"] = ci_hi
    except Exception:
        result[f"auc_{name}_ci_lower"] = result[f"auc_{name}"]
        result[f"auc_{name}_ci_upper"] = result[f"auc_{name}"]

    # Sensitivity at fixed specificity (100%, 99%, 95%)
    fpr, tpr, thresholds = roc_curve(y, oof_probs)
    zero_fpr_mask = fpr == 0.0
    if zero_fpr_mask.any():
        sens_100 = float(tpr[zero_fpr_mask][-1])
        thresh_100 = float(thresholds[zero_fpr_mask][-1])
    else:
        sens_100, thresh_100 = 0.0, float("inf")

    result[f"{name}_sensitivity_at_100spec"] = sens_100
    result[f"{name}_threshold_at_100spec"] = thresh_100
    result[f"{name}_n_detected_at_100spec"] = int(round(sens_100 * y.sum()))
    result[f"{name}_n_total_positive"] = int(y.sum())

    for target_spec, label in [(0.99, "99spec"), (0.95, "95spec")]:
        target_fpr = 1.0 - target_spec
        valid = fpr <= target_fpr
        result[f"{name}_sensitivity_at_{label}"] = (
            float(tpr[valid][-1]) if valid.any() else 0.0
        )

    # Healthy-normal specificity
    if sample_labels is not None:
        healthy_mask = np.array([lbl == "Healthy Normal" for lbl in sample_labels])
        if healthy_mask.any():
            max_healthy = float(oof_probs[healthy_mask].max())
            pos_mask = y == 1
            n_pos = int(pos_mask.sum())
            if n_pos > 0:
                detected = int((oof_probs[pos_mask] > max_healthy).sum())
            else:
                detected = 0
            result[f"{name}_sensitivity_at_100spec_healthy"] = detected / max(1, n_pos)
            result[f"{name}_threshold_at_100spec_healthy"] = max_healthy
            result[f"{name}_n_detected_at_100spec_healthy"] = detected

    # Optimal threshold + classification metrics
    threshold = _optimal_threshold(y, oof_probs)
    result[f"{name}_optimal_threshold"] = threshold
    cls_rep, cm = _classification_metrics(y, oof_probs, threshold=threshold)
    result[f"{name}_classification_report"] = cls_rep
    result[f"{name}_confusion_matrix"] = cm

    # Per-fold AUC tracking
    if fold_assignment is not None:
        fold_aucs = []
        for k in sorted(set(fold_assignment)):
            mask = np.array(fold_assignment) == k
            if len(np.unique(y[mask])) == 2:
                try:
                    fold_aucs.append(float(roc_auc_score(y[mask], oof_probs[mask])))
                except ValueError:
                    pass
        if fold_aucs:
            result[f"{name}_fold_aucs"] = fold_aucs
            result[f"{name}_auc_std"] = float(np.std(fold_aucs))

    return result


# ── Universal Model Evaluator ─────────────────────────────────────────────────


def evaluate_model(
    model,
    X: np.ndarray,
    y: np.ndarray,
    cv: StratifiedKFold,
    name: str,
    feature_names: list[str] | None = None,
    refit: bool = True,
    compute_shap: bool = False,
    shap_samples: int = 500,
    random_state: int = 42,
    sample_labels: np.ndarray | None = None,
) -> tuple[dict, object | None]:
    """Evaluate any sklearn-compatible model via stratified cross-validation.

    Shared primitive used by cpu_models(), gpu_models(), and multimodal_eval().
    The model must implement fit() and predict_proba().

    Args:
        model: sklearn-compatible estimator (Pipeline, RF, XGB, TabPFN, etc.).
        X: Feature matrix, shape (n_samples, n_features).
        y: Binary labels (0/1), shape (n_samples,).
        cv: Pre-configured StratifiedKFold splitter.
        name: Prefix for all result keys (e.g. "lr", "rf", "tabpfn").
        feature_names: Optional feature names for importance extraction.
        refit: If True, refit model on full data after CV.
        compute_shap: If True, compute SHAP values (requires refit=True).
        shap_samples: Max samples for SHAP computation.
        random_state: Random seed for bootstrap CI.
        sample_labels: Original 4-tier labels (e.g. "Healthy Normal",
            "Possible ctDNA+") for computing sensitivity@100%spec on
            healthy normals only. Shape (n_samples,). Optional.

    Returns:
        (result_dict, fitted_model_or_None). result_dict keys are prefixed
        with ``name`` (e.g. ``auc_rf``, ``rf_oof_probs``, ``rf_fold_aucs``,
        ``rf_sensitivity_at_100spec``).
    """
    import time

    result = {}

    # (Dead `classes_` block removed — GPUModelCVAdapter handles this now)

    # 1. Out-of-fold predictions via cross-validation
    oof_probs = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]

    # 2. Core AUC metric
    auc_val = roc_auc_score(y, oof_probs)
    result[f"auc_{name}"] = auc_val
    result[f"{name}_oof_probs"] = np.round(oof_probs, 6).tolist()

    # 3. Bootstrap 95% CI
    ci_lo, ci_hi = _bootstrap_auc(y, oof_probs, seed=random_state)
    result[f"auc_{name}_ci_lower"] = ci_lo
    result[f"auc_{name}_ci_upper"] = ci_hi

    # 3b. Sensitivity at fixed specificity operating points.
    # Clinically: "at the threshold where no healthy person is called
    # positive, how many cancers do we detect?"
    fpr, tpr, thresholds = roc_curve(y, oof_probs)

    # Primary: sensitivity @ 100% specificity (FPR = 0)
    zero_fpr_mask = fpr == 0.0
    if zero_fpr_mask.any():
        sens_100 = float(tpr[zero_fpr_mask][-1])
        thresh_100 = float(thresholds[zero_fpr_mask][-1])
    else:
        sens_100 = 0.0
        thresh_100 = float("inf")

    result[f"{name}_sensitivity_at_100spec"] = sens_100
    result[f"{name}_threshold_at_100spec"] = thresh_100
    result[f"{name}_n_detected_at_100spec"] = int(round(sens_100 * y.sum()))
    result[f"{name}_n_total_positive"] = int(y.sum())

    # Secondary: sensitivity @ 99% and 95% specificity
    for target_spec, spec_label in [(0.99, "99spec"), (0.95, "95spec")]:
        target_fpr = 1.0 - target_spec
        valid_mask = fpr <= target_fpr
        if valid_mask.any():
            result[f"{name}_sensitivity_at_{spec_label}"] = float(tpr[valid_mask][-1])
        else:
            result[f"{name}_sensitivity_at_{spec_label}"] = 0.0

    # Healthy-normal-only specificity: threshold set by the max predicted
    # probability among Healthy Normal samples (strictest clinical bar).
    if sample_labels is not None:
        healthy_mask = np.array([lbl == "Healthy Normal" for lbl in sample_labels])
        if healthy_mask.any():
            healthy_probs = oof_probs[healthy_mask]
            max_healthy_prob = float(healthy_probs.max())
            positive_mask = y == 1
            n_pos = int(positive_mask.sum())
            if n_pos > 0:
                detected = int((oof_probs[positive_mask] > max_healthy_prob).sum())
                sens_healthy = detected / n_pos
            else:
                detected = 0
                sens_healthy = 0.0
            result[f"{name}_sensitivity_at_100spec_healthy"] = float(sens_healthy)
            result[f"{name}_threshold_at_100spec_healthy"] = max_healthy_prob
            result[f"{name}_n_detected_at_100spec_healthy"] = detected
            log.info(
                "clinical_sensitivity_computed",
                model=name,
                sens_100spec=f"{sens_100:.3f}",
                sens_100spec_healthy=f"{sens_healthy:.3f}",
                n_detected_healthy=detected,
                n_total_positive=n_pos,
                n_healthy=int(healthy_mask.sum()),
            )
        else:
            log.debug("no_healthy_normals_in_sample_labels", model=name)
    else:
        log.debug(
            "sample_labels_not_provided",
            model=name,
            impact="sensitivity_at_100spec_healthy not computed",
        )

    # 4. Optimal threshold + classification metrics
    threshold = _optimal_threshold(y, oof_probs)
    result[f"{name}_optimal_threshold"] = threshold
    cls_rep, cm = _classification_metrics(y, oof_probs, threshold=threshold)
    result[f"{name}_classification_report"] = cls_rep
    result[f"{name}_confusion_matrix"] = cm

    # 5. Per-fold AUC tracking
    try:
        fold_aucs = _per_fold_auc(y, oof_probs, cv, X)
        if fold_aucs:
            result[f"{name}_fold_aucs"] = fold_aucs
            result[f"{name}_auc_std"] = float(np.std(fold_aucs))
            log.debug(
                "fold_aucs_computed",
                model=name,
                mean=f"{np.mean(fold_aucs):.3f}",
                std=f"{np.std(fold_aucs):.3f}",
            )
    except Exception as e:
        log.warning("fold_auc_tracking_failed", model=name, error=str(e))

    # 6. Precision-Recall curve
    pr_dict, avg_prec = _pr_curve(y, oof_probs)
    if pr_dict is not None:
        result[f"{name}_pr_curve"] = pr_dict
        result[f"{name}_avg_precision"] = avg_prec

    # 7. Calibration curve
    cal = _calibration(y, oof_probs)
    if cal is not None:
        result[f"{name}_calibration"] = cal

    # 8. Refit on full data + training time
    fitted = None
    if refit:
        t0 = time.perf_counter()
        model.fit(X, y)
        result[f"{name}_training_time_sec"] = time.perf_counter() - t0
        fitted = model

    # 9. Feature importances (model-specific)
    if fitted is not None and feature_names is not None:
        result[f"{name}_feature_importances"] = _extract_importances(
            fitted, feature_names, name
        )

    return result, fitted


def evaluate_holdout(
    fitted_model,
    X_test: np.ndarray,
    y_test: np.ndarray,
    name: str,
    sample_labels: np.ndarray | None = None,
) -> dict:
    """Evaluate a fitted model on the holdout test set.

    This is called AFTER ``evaluate_model()`` has refitted the model on
    the full training set. The holdout set was never seen during CV or
    feature selection, providing an unbiased estimate of generalization.

    Args:
        fitted_model: Model already fitted on training data.
        X_test: Holdout feature matrix.
        y_test: Holdout binary labels.
        name: Model name prefix for result keys.
        sample_labels: Original label strings for the holdout set
            (for healthy-normal specificity).

    Returns:
        Dict with holdout metrics, all prefixed with ``holdout_{name}_``.
    """
    result = {}
    prefix = f"holdout_{name}"

    if len(np.unique(y_test)) < 2:
        log.warning(
            "holdout_single_class",
            model=name,
            n_test=len(y_test),
            y_unique=np.unique(y_test).tolist(),
        )
        result[f"{prefix}_error"] = "single_class_in_holdout"
        return result

    try:
        probs = fitted_model.predict_proba(X_test)[:, 1]
    except Exception as e:
        log.error("holdout_predict_failed", model=name, error=str(e))
        result[f"{prefix}_error"] = str(e)
        return result

    # AUC
    auc_val = roc_auc_score(y_test, probs)
    result[f"{prefix}_auc"] = auc_val

    # Bootstrap CI
    ci_lo, ci_hi = _bootstrap_auc(y_test, probs, seed=42)
    result[f"{prefix}_auc_ci_lower"] = ci_lo
    result[f"{prefix}_auc_ci_upper"] = ci_hi

    # Sensitivity @ fixed specificity
    fpr, tpr, thresholds = roc_curve(y_test, probs)
    zero_fpr_mask = fpr == 0.0
    sens_100 = float(tpr[zero_fpr_mask][-1]) if zero_fpr_mask.any() else 0.0
    result[f"{prefix}_sensitivity_at_100spec"] = sens_100
    result[f"{prefix}_n_detected_at_100spec"] = int(round(sens_100 * y_test.sum()))
    result[f"{prefix}_n_total_positive"] = int(y_test.sum())
    result[f"{prefix}_n_total_negative"] = int((y_test == 0).sum())

    for target_spec, spec_label in [(0.99, "99spec"), (0.95, "95spec")]:
        target_fpr = 1.0 - target_spec
        valid_mask = fpr <= target_fpr
        if valid_mask.any():
            result[f"{prefix}_sensitivity_at_{spec_label}"] = float(tpr[valid_mask][-1])
        else:
            result[f"{prefix}_sensitivity_at_{spec_label}"] = 0.0

    # Healthy-normal specificity on holdout
    if sample_labels is not None:
        healthy_mask = np.array([lbl == "Healthy Normal" for lbl in sample_labels])
        if healthy_mask.any():
            max_healthy_prob = float(probs[healthy_mask].max())
            positive_mask = y_test == 1
            n_pos = int(positive_mask.sum())
            if n_pos > 0:
                detected = int((probs[positive_mask] > max_healthy_prob).sum())
                result[f"{prefix}_sensitivity_at_100spec_healthy"] = detected / n_pos
            else:
                result[f"{prefix}_sensitivity_at_100spec_healthy"] = 0.0

    # Classification metrics at optimal threshold
    threshold = _optimal_threshold(y_test, probs)
    result[f"{prefix}_optimal_threshold"] = threshold
    cls_rep, cm = _classification_metrics(y_test, probs, threshold=threshold)
    result[f"{prefix}_classification_report"] = cls_rep
    result[f"{prefix}_confusion_matrix"] = cm

    log.info(
        "holdout_evaluation_complete",
        model=name,
        holdout_auc=f"{auc_val:.3f}",
        holdout_sens_100spec=f"{sens_100:.3f}",
        n_test=len(y_test),
        n_positive=int(y_test.sum()),
    )

    return result


def _extract_importances(
    fitted_model, feature_names: list[str], model_name: str
) -> dict | list | None:
    """Extract feature importances from a fitted model.

    Supports:
    - Tree-based models (RF, XGB): feature_importances_ attribute
    - Linear models / Pipelines: coef_ attribute (abs values)
    - Others: returns None (no silent failure)
    """
    try:
        # Tree-based models
        if hasattr(fitted_model, "feature_importances_"):
            return dict(zip(feature_names, fitted_model.feature_importances_.tolist()))

        # Pipeline: check last step
        if hasattr(fitted_model, "named_steps"):
            last_step = list(fitted_model.named_steps.values())[-1]
            if hasattr(last_step, "coef_"):
                coefs = np.abs(last_step.coef_[0])
                return dict(zip(feature_names, coefs.tolist()))
            if hasattr(last_step, "feature_importances_"):
                return dict(zip(feature_names, last_step.feature_importances_.tolist()))

        # Direct linear model
        if hasattr(fitted_model, "coef_"):
            coefs = np.abs(fitted_model.coef_[0])
            return dict(zip(feature_names, coefs.tolist()))

    except Exception as e:
        log.warning(
            "feature_importance_extraction_failed", model=model_name, error=str(e)
        )

    return None


def cpu_models(
    X: np.ndarray,  # shape (n_samples, n_sub_metrics)
    y: np.ndarray,  # binary labels (1 = positive class)
    feature_names: list[str] | None = None,
    cancer_types: np.ndarray | None = None,
    assays: np.ndarray | None = None,
    n_folds: int = 5,
    random_state: int = 42,
    compute_shap: bool = False,
    shap_samples: int = 500,
    sample_labels: np.ndarray | None = None,
    per_fold_features: dict | None = None,
    fold_assignment: list | np.ndarray | None = None,
) -> tuple[dict, object, object, object]:
    """Train LR, RF, and XGB on a feature matrix with stratified CV.

    Delegates per-model evaluation to ``evaluate_model()``, then adds
    cross-model diagnostics (AUC deltas, top features, threshold sweep,
    DCA, feature stability, subgroup analysis).

    When ``per_fold_features`` and ``fold_assignment`` are provided
    (from nested CV ablation), bypasses ``evaluate_model()`` and uses
    manual fold iteration with per-fold feature filtering.

    Args:
        sample_labels: Original 4-tier labels for computing healthy-normal
            specificity. Passed to ``evaluate_model()``.
        per_fold_features: Dict from best_subset.json with per-model
            per-fold feature lists. Structure:
            ``{model_name: {"folds": {"0": {"features": [...]}, ...}}}``.
            When ``None``, uses standard path (all features, all folds).
        fold_assignment: Per-sample fold index array from ablation.
            Must align with the samples in X/y.

    Returns (results_dict, lr_pipeline, rf_model, xgb_model).

    Audit fixes preserved:
      - C-01: LR uses Pipeline(scaler+lr) to prevent data leakage
      - C-02: Subgroup metrics use out-of-fold predictions (unbiased)
      - H-01: LR has class_weight="balanced", XGB has scale_pos_weight
      - M-02: Bootstrap 95% CI on AUC values
    """
    from sklearn.metrics import confusion_matrix

    try:
        from xgboost import XGBClassifier
    except Exception:
        # XGBoost may fail with XGBoostError if libomp is missing (macOS)
        XGBClassifier = None

    results = {}
    try:
        y = y.astype(int)
        if len(np.unique(y)) < 2:
            log.warning("single_class_y", y_unique=np.unique(y).tolist())
            return {"error": "Only one class present in target array"}, None, None, None

        # Guard for small groups where n_splits doesn't work
        min_class_counts = np.bincount(y).min()
        folds = min(n_folds, min_class_counts)
        if folds < 2:
            return (
                {"error": "Not enough samples per class for Stratified CV (<2)"},
                None,
                None,
                None,
            )

        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
        results["cv_folds_actual"] = folds

        # ── NESTED CV PATH: per-fold feature subsets from ablation ──
        if per_fold_features is not None and fold_assignment is not None:
            fold_arr = np.array(fold_assignment)
            log.info(
                "cpu_models_nested_cv",
                n_folds=folds,
                models=list(per_fold_features.keys()),
            )

            # Build model definitions matching standard path hyperparams
            model_defs = {
                "lr": lambda: Pipeline(
                    [
                        ("scaler", StandardScaler()),
                        (
                            "lr",
                            LogisticRegression(
                                max_iter=1000,
                                random_state=random_state,
                                class_weight="balanced",
                            ),
                        ),
                    ]
                ),
                "rf": lambda: RandomForestClassifier(
                    n_estimators=100,
                    max_depth=5,
                    min_samples_leaf=max(1, min(10, len(y) // 10)),
                    random_state=random_state,
                    class_weight="balanced",
                ),
            }
            # XGB
            try:
                from xgboost import XGBClassifier as _XGB

                pos_w = len(y[y == 0]) / max(1, len(y[y == 1]))
                model_defs["xgb"] = lambda: _XGB(
                    n_estimators=100,
                    max_depth=5,
                    learning_rate=0.1,
                    random_state=random_state,
                    eval_metric="logloss",
                    scale_pos_weight=pos_w,
                )
            except Exception:
                pass

            lr_fitted, rf_fitted, xgb_fitted = None, None, None

            for model_name, model_factory in model_defs.items():
                if model_name not in per_fold_features:
                    log.debug(
                        "nested_cv_skip_model",
                        model=model_name,
                        reason="not_in_per_fold_features",
                    )
                    continue

                model_fold_data = per_fold_features[model_name].get("folds", {})
                oof_probs = np.full(len(y), np.nan)
                last_fitted = None

                for fold_k in sorted(set(fold_arr)):
                    train_mask = fold_arr != fold_k
                    test_mask = fold_arr == fold_k

                    fold_key = str(fold_k)
                    if fold_key in model_fold_data:
                        fold_feats = model_fold_data[fold_key].get("features", [])
                    else:
                        # Fallback: use all features if this fold is missing
                        fold_feats = feature_names if feature_names else []

                    # Map feature names to column indices
                    feat_idx = [
                        feature_names.index(f)  # type: ignore[union-attr]
                        for f in fold_feats
                        if f in (feature_names or [])
                    ]
                    if not feat_idx:
                        feat_idx = list(range(X.shape[1]))  # fallback to all

                    X_tr = X[train_mask][:, feat_idx]
                    X_te = X[test_mask][:, feat_idx]

                    m = model_factory()
                    m.fit(X_tr, y[train_mask])

                    if hasattr(m, "predict_proba"):
                        oof_probs[test_mask] = m.predict_proba(X_te)[:, 1]
                    elif hasattr(m, "decision_function"):
                        oof_probs[test_mask] = m.decision_function(X_te)

                    last_fitted = m

                # Verify full OOF coverage
                nan_count = int(np.isnan(oof_probs).sum())
                if nan_count > 0:
                    log.warning("nested_cv_oof_gaps", model=model_name, n_nan=nan_count)
                    oof_probs = np.nan_to_num(oof_probs, nan=0.5)

                # Compute standard metrics from stitched OOF
                model_res = _compute_oof_metrics(
                    y,
                    oof_probs,
                    model_name,
                    feature_names=feature_names,
                    sample_labels=sample_labels,
                    random_state=random_state,
                    fold_assignment=fold_arr,
                )
                results.update(model_res)

                # Refit on full data using most-common features
                all_fold_feats = [
                    model_fold_data[str(k)].get("features", [])
                    for k in sorted(set(fold_arr))
                    if str(k) in model_fold_data
                ]
                if all_fold_feats:
                    from collections import Counter

                    feat_counts = Counter(f for flist in all_fold_feats for f in flist)
                    # Features appearing in majority of folds
                    majority = len(all_fold_feats) // 2 + 1
                    refit_feats = [
                        f
                        for f, c in feat_counts.most_common()
                        if c >= majority and f in (feature_names or [])
                    ]
                    if not refit_feats:
                        refit_feats = feature_names or []
                else:
                    refit_feats = feature_names or []

                refit_idx = [
                    feature_names.index(f)  # type: ignore[union-attr]
                    for f in refit_feats
                    if f in (feature_names or [])
                ]
                if refit_idx:
                    final_model = model_factory()
                    final_model.fit(X[:, refit_idx], y)
                else:
                    final_model = last_fitted

                # Store refit feature names for holdout slicing (GAP 1 fix)
                results[f"{model_name}_refit_features"] = refit_feats

                if model_name == "lr":
                    lr_fitted = final_model
                elif model_name == "rf":
                    rf_fitted = final_model
                elif model_name == "xgb":
                    xgb_fitted = final_model

                log.info(
                    "nested_cv_model_done",
                    model=model_name,
                    auc=f"{results.get(f'auc_{model_name}', 0):.4f}",
                    n_refit_features=len(refit_feats),
                )

            # ── OOF labels ──
            results["oof_labels"] = y.tolist()
            results["nested_cv"] = True

            # ── AUC deltas (same as standard path) ──
            if "auc_rf" in results and "auc_lr" in results:
                results["auc_delta_rf_lr"] = results["auc_rf"] - results["auc_lr"]
            if "auc_xgb" in results and "auc_rf" in results:
                results["auc_delta_xgb_rf"] = results["auc_xgb"] - results["auc_rf"]

            return results, lr_fitted, rf_fitted, xgb_fitted

        # ── Logistic Regression (Pipeline prevents scaler leakage, C-01) ──
        lr_pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "lr",
                    LogisticRegression(
                        max_iter=1000,
                        random_state=random_state,
                        class_weight="balanced",
                    ),
                ),
            ]
        )
        lr_res, lr_pipe = evaluate_model(
            lr_pipe,
            X,
            y,
            cv,
            "lr",
            feature_names=feature_names,
            random_state=random_state,
            sample_labels=sample_labels,
        )
        results.update(lr_res)
        results["lr_coef_direction"] = (
            "positive" if lr_pipe.named_steps["lr"].coef_[0].mean() > 0 else "negative"
        )

        # ── Random Forest ──
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            min_samples_leaf=max(1, min(10, len(y) // 10)),
            random_state=random_state,
            class_weight="balanced",
        )
        rf_res, rf = evaluate_model(
            rf,
            X,
            y,
            cv,
            "rf",
            feature_names=feature_names,
            random_state=random_state,
            sample_labels=sample_labels,
        )
        results.update(rf_res)

        # ── XGBoost ──
        xgb = None
        if XGBClassifier is not None:
            pos_weight = len(y[y == 0]) / max(1, len(y[y == 1]))
            xgb_model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=random_state,
                eval_metric="logloss",
                scale_pos_weight=pos_weight,
            )
            try:
                xgb_res, xgb = evaluate_model(
                    xgb_model,
                    X,
                    y,
                    cv,
                    "xgb",
                    feature_names=feature_names,
                    random_state=random_state,
                    sample_labels=sample_labels,
                )
                results.update(xgb_res)
            except Exception as e:
                log.warning("xgboost_cv_failed", error=str(e))
                xgb = None

        # ── OOF labels (needed by report template for consistent ROC plots) ──
        results["oof_labels"] = y.tolist()

        # ── AUC deltas ──
        results["auc_delta_rf_lr"] = results["auc_rf"] - results["auc_lr"]
        if "auc_xgb" in results:
            results["auc_delta_xgb_rf"] = results["auc_xgb"] - results["auc_rf"]

        # ── Top features (post-CV, from RF importances) ──
        if feature_names is not None and isinstance(
            results.get("rf_feature_importances"), dict
        ):
            sorted_feats = sorted(
                results["rf_feature_importances"].items(),
                key=lambda x: x[1],
                reverse=True,
            )
            results["top_features"] = [f[0] for f in sorted_feats[:10]]

        # ── Feature stability across CV folds ──
        if feature_names is not None and rf is not None:
            try:
                from collections import Counter
                from sklearn.base import clone

                fold_top10 = Counter()
                for train_idx, _ in cv.split(X, y):
                    fold_rf = clone(rf).fit(X[train_idx], y[train_idx])
                    top10_idx = np.argsort(fold_rf.feature_importances_)[-10:]
                    for idx in top10_idx:
                        fold_top10[feature_names[idx]] += 1
                results["feature_stability"] = {
                    k: round(v / folds, 2) for k, v in fold_top10.items()
                }
            except Exception as e:
                log.warning("feature_stability_failed", error=str(e))

        # ── Threshold sensitivity sweep (RF) ──
        rf_probs = np.array(results.get("rf_oof_probs", []))
        if len(rf_probs) > 0:
            try:
                thresholds_sweep = np.linspace(0.01, 0.99, 50)
                sweep_data = []
                for t in thresholds_sweep:
                    y_pred = (rf_probs >= t).astype(int)
                    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
                    sweep_data.append(
                        {
                            "threshold": round(float(t), 3),
                            "sensitivity": (
                                float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
                            ),
                            "specificity": (
                                float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
                            ),
                            "ppv": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
                        }
                    )
                results["rf_threshold_sweep"] = sweep_data
            except Exception as e:
                log.warning("threshold_sweep_failed", error=str(e))

        # ── Decision Curve Analysis ──
        try:
            model_probs = {"rf": rf_probs}
            if "xgb_oof_probs" in results:
                model_probs["xgb"] = np.array(results["xgb_oof_probs"])
            for prefix, probs in model_probs.items():
                if len(probs) > 0:
                    results[f"{prefix}_dca"] = decision_curve_analysis(y, probs)
        except Exception as e:
            log.warning("dca_failed", error=str(e))

        # ── Subgroup Analysis using OOF predictions (C-02: unbiased) ──
        rf_opt = results.get("rf_optimal_threshold", 0.5)
        oof_preds = {}
        if len(rf_probs) > 0:
            oof_preds["rf"] = (rf_probs >= rf_opt).astype(int)
        if "xgb_oof_probs" in results:
            xgb_thresh = results.get("xgb_optimal_threshold", 0.5)
            xgb_probs_arr = np.array(results["xgb_oof_probs"])
            oof_preds["xgb"] = (xgb_probs_arr >= xgb_thresh).astype(int)

        # Cancer type subgroup analysis (Top 10)
        if cancer_types is not None:
            import pandas as pd

            c_stats = []
            c_series = pd.Series(cancer_types)
            top_10 = c_series.value_counts().nlargest(10).index

            for c_type in top_10:
                mask = cancer_types == c_type
                n_samp = mask.sum()
                if n_samp < 5:
                    continue
                res_sub = _subgroup_metrics(mask, y, oof_preds)
                if res_sub:
                    c_stats.append(
                        {
                            "cancer_type": c_type,
                            "n_samples": int(n_samp),
                            "n_positives": int(y[mask].sum()),
                            **res_sub,
                        }
                    )
            results["cancer_type_stats"] = c_stats

        # Assay subgroup analysis
        if assays is not None:
            import pandas as pd

            a_stats = []
            a_series = pd.Series(assays)
            for a_type in a_series.dropna().unique():
                mask = assays == a_type
                n_samp = mask.sum()
                if n_samp < 5:
                    continue
                res_sub = _subgroup_metrics(mask, y, oof_preds)
                if res_sub:
                    a_stats.append(
                        {
                            "assay": a_type,
                            "n_samples": int(n_samp),
                            "n_positives": int(y[mask].sum()),
                            **res_sub,
                        }
                    )
            results["assay_stats"] = a_stats

        return results, lr_pipe, rf, xgb
    except Exception as e:
        import traceback

        log.error("cpu_models_failed", error=str(e), trace=traceback.format_exc())
        return {"error": str(e)}, None, None, None


# Backward compatibility alias — existing code and tests use single_feature_model
single_feature_model = cpu_models


# ── GPU Foundation Model Support ──────────────────────────────────────────────


class GPUModelCVAdapter(BaseEstimator, ClassifierMixin):
    """sklearn-compatible wrapper for GPU foundation models.

    Sidesteps two TabPFN sklearn-compatibility issues:

    1. ``classes_`` is a read-only ``@property`` on ``FinetunedTabPFNClassifier``
       that raises ``AttributeError`` when unfitted.  ``cross_val_predict``
       tries to read ``classes_`` on the *fitted clone*, but ``sklearn.clone()``
       copies the unfitted object first and the property breaks. This adapter
       exposes ``classes_`` as a plain attribute set during ``fit()``.

    2. ``sklearn.base.clone()`` is unreliable for TabPFN
       (PriorLabs/TabPFN#327).  By accepting a ``model_factory`` callable,
       each ``fit()`` creates a fresh instance without relying on clone.

    Attributes:
        model_factory: Zero-argument callable returning a fresh model instance.
        name: Human-readable model name for logging.
        model_: The fitted model instance (set after ``fit()``).
        classes_: Unique sorted class labels (set after ``fit()``).
    """

    def __init__(self, model_factory: Callable, name: str = "gpu_model"):
        self.model_factory = model_factory
        self.name = name

    def __getstate__(self) -> dict:
        """Exclude ``model_factory`` (a lambda closure) from pickle serialization.

        The factory is only needed for training (creating fresh model instances
        during CV). Deserialized adapters are inference-only — ``predict_proba()``
        uses the already-fitted ``model_`` attribute.
        """
        state = self.__dict__.copy()
        state.pop("model_factory", None)
        return state

    def __setstate__(self, state: dict) -> None:
        """Restore adapter in inference-only mode (no ``model_factory``).

        Calling ``fit()`` on a deserialized adapter will raise ``TypeError``
        because ``model_factory`` is None. This is intentional — loaded models
        are for prediction/SHAP only.
        """
        self.__dict__.update(state)
        if "model_factory" not in self.__dict__:
            self.model_factory = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GPUModelCVAdapter":
        """Create a fresh model via factory and fit it.

        Args:
            X: Feature matrix (n_samples, n_features).
            y: Binary label array (n_samples,).

        Returns:
            self

        Raises:
            TypeError: If called on a deserialized adapter (``model_factory``
                is None). Loaded models are inference-only.
        """
        if self.model_factory is None:
            raise TypeError(
                f"Cannot fit() a deserialized {self.__class__.__name__} — "
                f"model_factory is not available. Loaded models are inference-only."
            )
        self.model_ = self.model_factory()
        self.model_.fit(X, y)
        self.classes_ = np.unique(y)
        log.info(
            "gpu_adapter_fit",
            model=self.name,
            n_samples=X.shape[0],
            n_features=X.shape[1],
            classes=self.classes_.tolist(),
        )
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities.

        Raises:
            NotFittedError: If ``fit()`` has not been called.
        """
        check_is_fitted(self, "model_")
        return self.model_.predict_proba(X)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels.

        Raises:
            NotFittedError: If ``fit()`` has not been called.
        """
        check_is_fitted(self, "model_")
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def _build_gpu_model(
    name: str,
    device: str = "cuda",
    finetune_epochs: int = 50,
    finetune_lr: float = 1e-5,
    random_state: int = 42,
) -> GPUModelCVAdapter | None:
    """Factory: instantiate a GPU foundation model wrapped in GPUModelCVAdapter.

    All models are returned inside a :class:`GPUModelCVAdapter` so that
    ``sklearn.cross_val_predict`` and ``sklearn.clone`` work correctly.

    Args:
        name: One of ``"tabpfn"`` (zero-shot), ``"tabpfn_ft"`` (fine-tuned),
            ``"tabicl"`` (zero-shot), or ``"tabicl_ft"`` (fine-tuned).
        device: PyTorch device string (default ``"cuda"``).
        finetune_epochs: Training epochs for fine-tuned variants (default 50,
            matches TabICL default; TabPFN has internal early stopping).
        finetune_lr: Learning rate for fine-tuned variants.
        random_state: Random seed for reproducibility (used by zero-shot
            variants and TabICL fine-tuning).

    Returns:
        A :class:`GPUModelCVAdapter` wrapping the requested model, or
        ``None`` if the required library cannot be imported.

    Versions:
        - TabPFN v8.0.3+: import from ``tabpfn.finetuning``
        - TabPFN v8.1+: ``epochs`` param; ``random_state`` removed from
          ``FinetunedTabPFNClassifier``
        - TabICL v2.1+: ``FinetunedTabICLClassifier`` class
    """
    _KNOWN = {"tabpfn", "tabpfn_ft", "tabicl", "tabicl_ft"}

    if name not in _KNOWN:
        log.error(
            "unknown_gpu_model",
            model=name,
            known=sorted(_KNOWN),
            hint="Check --gpu-models / params.gpu_models spelling",
        )
        return None

    # ── TabPFN zero-shot ──
    if name == "tabpfn":
        try:
            from tabpfn import TabPFNClassifier

            log.info("gpu_model_built", model=name, mode="zero-shot")
            return GPUModelCVAdapter(
                model_factory=lambda: TabPFNClassifier(
                    device=device, random_state=random_state
                ),
                name=name,
            )
        except ImportError as e:
            log.error(
                "gpu_model_import_failed",
                model=name,
                error=str(e),
                hint="Install with: pip install kreview[gpu]  # requires tabpfn>=8.0.3",
            )
            return None

    # ── TabPFN fine-tuned ──
    if name == "tabpfn_ft":
        try:
            from tabpfn.finetuning import FinetunedTabPFNClassifier

            # Capture args in closure defaults to avoid late-binding issues
            log.info(
                "gpu_model_built",
                model=name,
                mode="fine-tuned",
                epochs=finetune_epochs,
                lr=finetune_lr,
            )
            return GPUModelCVAdapter(
                model_factory=lambda _d=device, _e=finetune_epochs, _lr=finetune_lr: (
                    FinetunedTabPFNClassifier(device=_d, epochs=_e, learning_rate=_lr)
                ),
                name=name,
            )
        except ImportError as e:
            log.error(
                "gpu_model_import_failed",
                model=name,
                error=str(e),
                hint="Install with: pip install kreview[gpu]  # requires tabpfn>=8.0.3",
            )
            return None

    # ── TabICL zero-shot ──
    if name == "tabicl":
        try:
            from tabicl import TabICLClassifier

            log.info("gpu_model_built", model=name, mode="zero-shot")
            return GPUModelCVAdapter(
                model_factory=lambda: TabICLClassifier(random_state=random_state),
                name=name,
            )
        except ImportError as e:
            log.error(
                "gpu_model_import_failed",
                model=name,
                error=str(e),
                hint="Install with: pip install kreview[gpu]  # requires tabicl>=2.1",
            )
            return None

    # ── TabICL fine-tuned ──
    if name == "tabicl_ft":
        try:
            from tabicl import FinetunedTabICLClassifier

            log.info(
                "gpu_model_built",
                model=name,
                mode="fine-tuned",
                epochs=finetune_epochs,
                lr=finetune_lr,
            )
            return GPUModelCVAdapter(
                model_factory=lambda _e=finetune_epochs, _lr=finetune_lr, _rs=random_state: (
                    FinetunedTabICLClassifier(
                        epochs=_e, learning_rate=_lr, random_state=_rs
                    )
                ),
                name=name,
            )
        except ImportError as e:
            log.error(
                "gpu_model_import_failed",
                model=name,
                error=str(e),
                hint="Install with: pip install kreview[gpu]  # requires tabicl>=2.1",
            )
            return None

    # Unreachable — _KNOWN check above covers all cases
    return None  # pragma: no cover


def _compute_shap(
    fitted_model,
    X: np.ndarray,
    name: str,
    feature_names: list[str] | None = None,
    max_samples: int = 500,
    random_state: int = 42,
) -> dict | None:
    """Compute SHAP values using the best available method for the model type.

    Dispatches to:
    - ``shapiq.TabularExplainer`` for foundation models (TabPFN, TabICL)
    - ``shap.TreeExplainer`` for tree-based models (RF, XGBoost)
    - ``shap.PermutationExplainer`` as universal fallback

    Returns dict with shap_values (list) and feature_names, or None on failure.
    """
    X_shap = X[:max_samples] if len(X) > max_samples else X

    try:
        # GPU foundation models: use shapiq (remove-and-recontextualize strategy)
        _gpu_base_names = {"tabpfn", "tabpfn_ft", "tabicl", "tabicl_ft"}
        if name in _gpu_base_names:
            try:
                import shapiq

                explainer = shapiq.TabularExplainer(
                    model=fitted_model.predict_proba,
                    data=X_shap,
                    random_state=random_state,
                )
                sv = explainer.explain(X_shap)
                log.info("shapiq_computed", model=name, n_samples=len(X_shap))
                return {
                    "shap_values": (
                        sv.values.tolist()
                        if hasattr(sv.values, "tolist")
                        else sv.values
                    ),
                    "feature_names": feature_names,
                }
            except Exception as e:
                log.warning(
                    "shapiq_failed",
                    model=name,
                    error=str(e),
                    fallback="permutation",
                )
                # Fall through to PermutationExplainer below

        # Tree-based (RF, XGB)
        import shap

        if hasattr(fitted_model, "feature_importances_"):
            explainer = shap.TreeExplainer(fitted_model)
            sv = explainer.shap_values(X_shap)
            # TreeExplainer may return list of arrays (one per class)
            if isinstance(sv, list):
                sv = sv[1]  # positive class
            return {
                "shap_values": sv.tolist(),
                "feature_names": feature_names,
            }

        # Pipeline: try to get inner model
        if hasattr(fitted_model, "named_steps"):
            last = list(fitted_model.named_steps.values())[-1]
            if hasattr(last, "feature_importances_"):
                explainer = shap.TreeExplainer(last)
                # Transform X through pipeline steps before last

                X_transformed = X_shap
                for step_name, step in list(fitted_model.named_steps.items())[:-1]:
                    X_transformed = step.transform(X_transformed)
                sv = explainer.shap_values(X_transformed)
                if isinstance(sv, list):
                    sv = sv[1]
                return {
                    "shap_values": sv.tolist(),
                    "feature_names": feature_names,
                }

        # Fallback: PermutationExplainer (slow but universal)
        explainer = shap.PermutationExplainer(
            fitted_model.predict_proba, X_shap, seed=random_state
        )
        sv = explainer(X_shap)
        return {
            "shap_values": (
                sv.values[:, :, 1].tolist()
                if sv.values.ndim == 3
                else sv.values.tolist()
            ),
            "feature_names": feature_names,
        }

    except Exception as e:
        log.warning("shap_computation_failed", model=name, error=str(e))
        return None


def gpu_models(
    X: np.ndarray,
    y: np.ndarray,
    feature_names: list[str] | None = None,
    cancer_types: np.ndarray | None = None,
    assays: np.ndarray | None = None,
    n_folds: int = 5,
    random_state: int = 42,
    models: tuple[str, ...] = ("tabpfn",),
    device: str = "cuda",
    finetune_epochs: int = 50,
    finetune_lr: float = 1e-5,
    compute_shap: bool = False,
    shap_samples: int = 500,
    sample_labels: np.ndarray | None = None,
    max_gpu_features: int | None = None,
    eval_stats: pd.DataFrame | None = None,
    per_fold_features: dict | None = None,
    fold_assignment: list | np.ndarray | None = None,
) -> tuple[dict, dict[str, object]]:
    """Train GPU foundation models on a feature matrix.

    Same output schema as ``cpu_models()`` for each model, using the shared
    ``evaluate_model()`` primitive.  Each model name encodes its variant:
    ``"tabpfn"`` = zero-shot, ``"tabpfn_ft"`` = fine-tuned, etc.

    Args:
        X: Feature matrix, shape (n_samples, n_features).
        y: Binary labels (0/1).
        feature_names: Optional feature names.
        cancer_types: Optional cancer type array for subgroup analysis.
        assays: Optional assay array for subgroup analysis.
        n_folds: Number of CV folds.
        random_state: Random seed.
        models: Tuple of GPU model names. Valid values:
            ``"tabpfn"`` (zero-shot), ``"tabpfn_ft"`` (fine-tuned),
            ``"tabicl"`` (zero-shot), ``"tabicl_ft"`` (fine-tuned).
        device: PyTorch device (``"cuda"``, ``"cpu"``).
        finetune_epochs: Epochs for fine-tuned variants (default 50).
        finetune_lr: Learning rate for fine-tuned variants.
        compute_shap: If True, compute SHAP values.
        shap_samples: Max samples for SHAP computation.
        sample_labels: Original 4-tier labels for computing healthy-normal
            specificity. Passed to ``evaluate_model()``.
        max_gpu_features: If set and n_features > max_gpu_features,
            cap to the top N features using eval_stats scores.
            Default None (no cap). CLI default is 150.
        eval_stats: DataFrame from ``score_features()`` containing
            feature scoring columns. Used for intelligent feature
            capping when max_gpu_features is set.
        per_fold_features: Dict from best_subset.json with per-model
            per-fold feature lists (from ablation). When provided, uses
            manual fold iteration with per-fold feature filtering.
        fold_assignment: Per-sample fold index array from ablation.

    Returns:
        (results_dict, fitted_models_dict). fitted_models_dict maps
        model name to fitted model object.
    """
    results = {}
    fitted_models = {}

    # ── GPU feature capping (v0.0.16+) ──
    # TabPFN has practical limits on feature count (~150-200).
    # If max_gpu_features is set and we exceed it, select the top N
    # features by score (mutual_info > univariate_auc > variance).
    n_orig_features = X.shape[1]
    if max_gpu_features is not None and n_orig_features > max_gpu_features:
        if eval_stats is not None and feature_names is not None:
            # Score-based selection from eval_stats
            score_col = None
            for candidate in ["mutual_info", "univariate_auc"]:
                if candidate in eval_stats.columns:
                    score_col = candidate
                    break

            if score_col and "feature_column" in eval_stats.columns:
                # Match eval_stats features to current feature_names
                scored = eval_stats[
                    eval_stats["feature_column"].isin(feature_names)
                ].nlargest(max_gpu_features, score_col)
                selected = scored["feature_column"].tolist()
                keep_idx = [i for i, f in enumerate(feature_names) if f in selected]
            else:
                # Fallback: variance-based selection
                log.warning("gpu_cap_no_score_column", fallback="variance")
                variances = np.var(X, axis=0)
                keep_idx = np.argsort(variances)[-max_gpu_features:].tolist()
        else:
            # No eval_stats: variance-based fallback
            log.warning("gpu_cap_no_eval_stats", fallback="variance")
            variances = np.var(X, axis=0)
            keep_idx = np.argsort(variances)[-max_gpu_features:].tolist()

        keep_idx.sort()  # Preserve original column order
        X = X[:, keep_idx]
        if feature_names is not None:
            feature_names = [feature_names[i] for i in keep_idx]

        log.info(
            "gpu_features_capped",
            original=n_orig_features,
            capped_to=X.shape[1],
            max_gpu_features=max_gpu_features,
            method="score_based" if eval_stats is not None else "variance",
        )
        results["gpu_feature_cap_applied"] = True
        results["gpu_feature_cap_original"] = n_orig_features
        results["gpu_feature_cap_final"] = X.shape[1]
        results["gpu_feature_cap_indices"] = keep_idx  # Callers use to cap X_test
    else:
        results["gpu_feature_cap_applied"] = False

    try:
        y = y.astype(int)
        if len(np.unique(y)) < 2:
            log.warning("single_class_y", y_unique=np.unique(y).tolist())
            return {"error": "Only one class present in target array"}, {}

        min_class_counts = np.bincount(y).min()
        folds = min(n_folds, min_class_counts)
        if folds < 2:
            return (
                {"error": "Not enough samples per class for Stratified CV (<2)"},
                {},
            )

        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
        results["cv_folds_actual"] = folds

        # ── NESTED CV PATH: per-fold feature subsets from ablation ──
        if per_fold_features is not None and fold_assignment is not None:
            fold_arr = np.array(fold_assignment)
            log.info(
                "gpu_models_nested_cv",
                n_folds=folds,
                models=list(per_fold_features.keys()),
            )

            for model_name in models:
                if model_name not in per_fold_features:
                    log.debug("nested_cv_gpu_skip", model=model_name)
                    continue

                model_fold_data = per_fold_features[model_name].get("folds", {})
                oof_probs = np.full(len(y), np.nan)
                last_fitted = None

                for fold_k in sorted(set(fold_arr)):
                    train_mask = fold_arr != fold_k
                    test_mask = fold_arr == fold_k

                    fold_key = str(fold_k)
                    if fold_key in model_fold_data:
                        fold_feats = model_fold_data[fold_key].get("features", [])
                    else:
                        fold_feats = feature_names if feature_names else []

                    feat_idx = [
                        feature_names.index(f)  # type: ignore[union-attr]
                        for f in fold_feats
                        if f in (feature_names or [])
                    ]
                    if not feat_idx:
                        feat_idx = list(range(X.shape[1]))

                    # Apply GPU feature cap per-fold
                    if max_gpu_features and len(feat_idx) > max_gpu_features:
                        variances = np.var(X[train_mask][:, feat_idx], axis=0)
                        top_k = np.argsort(variances)[-max_gpu_features:]
                        feat_idx = [feat_idx[i] for i in top_k]

                    X_tr = X[train_mask][:, feat_idx]
                    X_te = X[test_mask][:, feat_idx]

                    try:
                        model = _build_gpu_model(
                            model_name,
                            device=device,
                            finetune_epochs=finetune_epochs,
                            finetune_lr=finetune_lr,
                            random_state=random_state,
                        )
                        if model is None:
                            continue
                        model.fit(X_tr, y[train_mask])
                        if hasattr(model, "predict_proba"):
                            oof_probs[test_mask] = model.predict_proba(X_te)[:, 1]
                        last_fitted = model
                    except Exception as e:
                        log.warning(
                            "nested_cv_gpu_fold_error",
                            model=model_name,
                            fold=fold_k,
                            error=str(e),
                        )

                nan_count = int(np.isnan(oof_probs).sum())
                if nan_count > 0:
                    log.warning(
                        "nested_cv_gpu_oof_gaps", model=model_name, n_nan=nan_count
                    )
                    oof_probs = np.nan_to_num(oof_probs, nan=0.5)

                model_res = _compute_oof_metrics(
                    y,
                    oof_probs,
                    model_name,
                    feature_names=feature_names,
                    sample_labels=sample_labels,
                    random_state=random_state,
                    fold_assignment=fold_arr,
                )
                results.update(model_res)
                if last_fitted is not None:
                    fitted_models[model_name] = last_fitted

                # Track features used by last fold's model for holdout (GAP 1)
                last_fold_key = str(max(set(fold_arr)))
                if last_fold_key in model_fold_data:
                    last_feats = model_fold_data[last_fold_key].get("features", [])
                else:
                    last_feats = feature_names or []
                results[f"{model_name}_refit_features"] = last_feats

                log.info(
                    "nested_cv_gpu_model_done",
                    model=model_name,
                    auc=f"{results.get(f'auc_{model_name}', 0):.4f}",
                    n_refit_features=len(last_feats),
                )

            results["oof_labels"] = y.tolist()
            results["nested_cv"] = True
            return results, fitted_models

        for model_name in models:
            model = _build_gpu_model(
                model_name,
                device=device,
                finetune_epochs=finetune_epochs,
                finetune_lr=finetune_lr,
                random_state=random_state,
            )
            if model is None:
                log.warning(
                    "gpu_model_skipped",
                    model=model_name,
                    reason="build failed — package not installed or import error",
                )
                results[f"{model_name}_error"] = (
                    f"{model_name} requested but failed to import. "
                    f"Install with: pip install kreview[gpu]"
                )
                print(
                    f"⚠ WARNING: GPU model '{model_name}' was requested but "
                    f"could not be imported. Skipping.",
                    flush=True,
                )
                continue

            try:
                model_res, fitted = evaluate_model(
                    model,
                    X,
                    y,
                    cv,
                    model_name,
                    feature_names=feature_names,
                    refit=True,
                    compute_shap=compute_shap,
                    shap_samples=shap_samples,
                    random_state=random_state,
                    sample_labels=sample_labels,
                )
                results.update(model_res)
                fitted_models[model_name] = fitted

                # SHAP via native dispatchers
                if compute_shap and fitted is not None:
                    shap_result = _compute_shap(
                        fitted,
                        X,
                        model_name,
                        feature_names,
                        shap_samples,
                        random_state=random_state,
                    )
                    if shap_result is not None:
                        results[f"{model_name}_shap"] = shap_result

                log.info(
                    "gpu_model_evaluated",
                    model=model_name,
                    auc=results.get(f"auc_{model_name}"),
                )
            except Exception as e:
                log.error(
                    "gpu_model_evaluation_failed",
                    model=model_name,
                    error=str(e),
                )
                # Record the error in results so it's visible in the JSON output.
                # Without this, GPU failures are silent — no AUC key, no error key.
                results[f"{model_name}_error"] = str(e)
                continue

        # OOF labels for downstream multimodal alignment
        results["oof_labels"] = y.tolist()

        # ── Subgroup Analysis (matching cpu_models pattern) ──
        # Use OOF predictions from successful GPU models for unbiased metrics
        oof_preds = {}
        for model_name in models:
            probs_key = f"{model_name}_oof_probs"
            if probs_key in results:
                thresh = results.get(f"{model_name}_optimal_threshold", 0.5)
                probs_arr = np.array(results[probs_key])
                oof_preds[model_name] = (probs_arr >= thresh).astype(int)

        if oof_preds:
            # Cancer type subgroup analysis
            if cancer_types is not None:
                c_stats = []
                c_series = pd.Series(cancer_types)
                top_10 = c_series.value_counts().nlargest(10).index
                for c_type in top_10:
                    mask = cancer_types == c_type
                    n_samp = mask.sum()
                    if n_samp < 5:
                        continue
                    res_sub = _subgroup_metrics(mask, y, oof_preds)
                    if res_sub:
                        c_stats.append(
                            {
                                "cancer_type": c_type,
                                "n_samples": int(n_samp),
                                "n_positives": int(y[mask].sum()),
                                **res_sub,
                            }
                        )
                results["cancer_type_stats"] = c_stats

            # Assay subgroup analysis
            if assays is not None:
                a_stats = []
                a_series = pd.Series(assays)
                for a_type in a_series.dropna().unique():
                    mask = assays == a_type
                    n_samp = mask.sum()
                    if n_samp < 5:
                        continue
                    res_sub = _subgroup_metrics(mask, y, oof_preds)
                    if res_sub:
                        a_stats.append(
                            {
                                "assay": a_type,
                                "n_samples": int(n_samp),
                                "n_positives": int(y[mask].sum()),
                                **res_sub,
                            }
                        )
                results["assay_stats"] = a_stats

        # Detect when ALL GPU models failed — the results dict will have
        # oof_labels but no auc_* keys, which is indistinguishable from
        # success without this explicit check.
        successful = [m for m in models if f"auc_{m}" in results]
        if not successful:
            failed_info = {m: results.get(f"{m}_error", "unknown") for m in models}
            log.warning(
                "gpu_no_models_succeeded",
                requested=list(models),
                errors=failed_info,
            )
            results["error"] = f"All GPU models failed: {list(models)}"

        return results, fitted_models
    except Exception as e:
        import traceback

        log.error("gpu_models_failed", error=str(e), trace=traceback.format_exc())
        return {"error": str(e)}, {}


# ── CPU+GPU JSON Merge Helpers ──────────────────────────────────────────────


def load_model_results(
    directory: Path,
    evaluator_name: str,
) -> dict | None:
    """Load and merge CPU + GPU model results for a single evaluator.

    Looks for:

    - ``{evaluator_name}_model_results.json`` (CPU results)
    - ``{evaluator_name}_gpu_model_results.json`` (merged GPU results)
    - ``{evaluator_name}_*_gpu_model_results.json`` (scattered per-model
      GPU results from NF scattering, e.g. ``FSC_tabpfn_gpu_model_results.json``)

    All found GPU files are merged into the CPU dict.  GPU keys take
    precedence for overlapping model keys; CPU metadata keys
    (``evaluator``, ``matrix_path``, etc.) are preserved.

    Used by report templates where the evaluator name is known.

    Args:
        directory: Directory containing the JSON result files.
        evaluator_name: Evaluator name (e.g., ``"FSCOnTarget"``).

    Returns:
        Merged dict, or None if no JSON exists for this evaluator.
    """
    directory = Path(directory)
    cpu_path = directory / f"{evaluator_name}_model_results.json"

    # Metadata keys that should come from CPU (canonical source).
    # GPU JSONs from different models may have different feature counts
    # (due to feature capping), so oof_sample_ids and oof_labels from
    # GPU are skipped to avoid length mismatches.
    _skip = {
        "evaluator",
        "matrix_path",
        "cv_folds_actual",
        "oof_sample_ids",
        "oof_labels",
    }

    merged = None

    # ── Load CPU results ──
    if cpu_path.exists():
        try:
            with open(cpu_path) as f:
                merged = json.load(f)
        except (json.JSONDecodeError, OSError) as exc:
            log.warning(
                "load_model_results_cpu_failed",
                evaluator=evaluator_name,
                path=str(cpu_path),
                error=str(exc),
            )

    # ── Discover all GPU JSON files ──
    # 1. Merged GPU JSON (from NF merge step or single GPU task)
    gpu_merged_path = directory / f"{evaluator_name}_gpu_model_results.json"
    # 2. Per-model scattered GPU JSONs (from NF scattering)
    per_model_gpu_paths = sorted(
        p
        for p in directory.glob(f"{evaluator_name}_*_gpu_model_results.json")
        if p != gpu_merged_path  # Avoid double-counting the merged file
    )

    all_gpu_paths = []
    if gpu_merged_path.exists():
        all_gpu_paths.append(gpu_merged_path)
    all_gpu_paths.extend(per_model_gpu_paths)

    # ── Merge GPU results into CPU dict ──
    gpu_files_loaded = 0
    for gpu_path in all_gpu_paths:
        try:
            with open(gpu_path) as f:
                gpu_data = json.load(f)
        except (json.JSONDecodeError, OSError) as exc:
            log.warning(
                "load_model_results_gpu_failed",
                evaluator=evaluator_name,
                path=str(gpu_path),
                error=str(exc),
            )
            continue

        gpu_files_loaded += 1
        if merged is not None:
            merged.update({k: v for k, v in gpu_data.items() if k not in _skip})
        else:
            # GPU-only (no CPU results) — use GPU data as base
            merged = gpu_data

    if gpu_files_loaded > 0:
        log.info(
            "load_model_results_gpu_merged",
            evaluator=evaluator_name,
            gpu_files=gpu_files_loaded,
            per_model_files=len(per_model_gpu_paths),
            total_keys=len(merged) if merged else 0,
        )

    return merged


def load_all_model_results(
    directory: Path,
) -> dict[str, dict]:
    """Load and merge all CPU + GPU model results from a directory.

    Scans for ``*_model_results.json`` and ``*_gpu_model_results.json``,
    groups by evaluator name, and merges GPU keys into CPU dicts.

    Used by scoreboard and multimodal engine for directory scanning.

    Args:
        directory: Directory containing model result JSON files.

    Returns:
        Dict keyed by evaluator name → merged model results dict.
        Empty dict if no results found.
    """
    directory = Path(directory)
    # Discover all evaluator names from both CPU and GPU JSONs
    evaluators = set()
    for f in directory.glob("*_model_results.json"):
        name = f.stem
        if name.endswith("_gpu_model_results"):
            evaluators.add(name.replace("_gpu_model_results", ""))
        else:
            evaluators.add(name.replace("_model_results", ""))

    if not evaluators:
        log.warning("load_all_model_results_empty", dir=str(directory))
        return {}

    results = {}
    for eval_name in sorted(evaluators):
        merged = load_model_results(directory, eval_name)
        if merged is not None:
            results[eval_name] = merged

    log.info(
        "load_all_model_results_complete",
        dir=str(directory),
        n_evaluators=len(results),
        evaluators=sorted(results.keys()),
    )
    return results


# ── Phase D: Multimodal Evaluation ──────────────────────────────────────────


def _load_per_evaluator_baselines(
    results_dir: str | Path,
) -> dict:
    """Load all ``*_model_results.json`` files from a directory.

    Extracts per-evaluator OOF probabilities, labels, sample IDs, and
    AUC baselines needed for multimodal stacking.

    Args:
        results_dir: Directory containing ``*_model_results.json`` files
            produced by ``kreview eval cpu`` or ``kreview eval gpu``.

    Returns:
        A dict keyed by evaluator name::

            {
                "FSCOnTarget": {
                    "models": ["lr", "rf", "xgb"],
                    "oof_probs": {"lr": [...], "rf": [...], ...},
                    "oof_labels": [...],
                    "oof_sample_ids": [...],
                    "aucs": {"lr": 0.85, "rf": 0.88, ...},
                    "best_model": "auc_rf",
                    "best_auc": 0.88,
                },
                ...
            }

    Raises:
        FileNotFoundError: If ``results_dir`` does not exist.
        ValueError: If no valid JSON files are found.
    """
    results_dir = Path(results_dir)
    if not results_dir.exists():
        raise FileNotFoundError(f"Results directory not found: {results_dir}")

    # Use merge helper to discover and merge CPU+GPU JSONs automatically
    all_results = load_all_model_results(results_dir)
    if not all_results:
        raise ValueError(f"No *_model_results.json files in {results_dir}")

    baselines = {}
    n_loaded, n_skipped = 0, 0

    for eval_name, data in all_results.items():

        # Skip error results
        if "error" in data:
            log.warning(
                "multimodal_skip_errored_evaluator",
                evaluator=eval_name,
                error=data["error"],
            )
            n_skipped += 1
            continue

        # Discover which models have OOF probs
        model_names = []
        oof_probs = {}
        aucs = {}
        for k, v in data.items():
            if k.endswith("_oof_probs") and isinstance(v, list):
                mn = k.replace("_oof_probs", "")
                model_names.append(mn)
                oof_probs[mn] = v
            if (
                k.startswith("auc_")
                and isinstance(v, (int, float))
                and not k.endswith("_ci_lower")
                and not k.endswith("_ci_upper")
            ):
                # Exclude CI bounds
                aucs[k.replace("auc_", "")] = v

        if not model_names:
            log.warning(
                "multimodal_no_oof_probs",
                evaluator=eval_name,
                keys_sample=list(data.keys())[:10],
            )
            n_skipped += 1
            continue

        # Extract labels and sample IDs
        oof_labels = data.get("oof_labels")
        oof_sample_ids = data.get("oof_sample_ids")

        if oof_labels is None:
            log.warning("multimodal_missing_oof_labels", evaluator=eval_name)
            n_skipped += 1
            continue

        baselines[eval_name] = {
            "models": model_names,
            "oof_probs": oof_probs,
            "oof_labels": oof_labels,
            "oof_sample_ids": oof_sample_ids,
            "aucs": aucs,
            "best_model": data.get("best_model"),
            "best_auc": data.get("best_auc"),
        }
        n_loaded += 1

    log.info(
        "multimodal_baselines_loaded",
        n_loaded=n_loaded,
        n_skipped=n_skipped,
        evaluators=sorted(baselines.keys()),
    )

    if not baselines:
        raise ValueError(
            f"No valid evaluator baselines found in {results_dir} "
            f"({n_skipped} skipped)"
        )

    return baselines


def _build_stacking_matrix(
    baselines: dict,
    *,
    model_filter: str | None = None,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Build a meta-feature matrix from OOF predictions across evaluators.

    Each column is ``{evaluator}_{model}`` containing that model's OOF
    probability for the given evaluator.  Rows are aligned by
    ``oof_sample_ids`` via outer-join so all samples are represented.

    Args:
        baselines: Output of :func:`_load_per_evaluator_baselines`.
        model_filter: If provided, only include this model's OOF probs
            (e.g. ``"rf"``).  None means include all available models.

    Returns:
        (stacking_df, y) where ``stacking_df`` has columns like
        ``FSCOnTarget_rf``, ``MdsOnTarget_lr``, etc., and ``y`` is the
        aligned binary label array.

    Raises:
        ValueError: If no stacking columns can be built (e.g. no
            evaluators have ``oof_sample_ids``).
    """
    frames = []
    evaluators_with_ids = 0
    evaluators_without_ids = 0

    for eval_name, info in sorted(baselines.items()):
        sample_ids = info.get("oof_sample_ids")
        if sample_ids is None:
            log.warning(
                "stacking_missing_sample_ids",
                evaluator=eval_name,
                hint="Run with oof_sample_ids for correct alignment",
            )
            evaluators_without_ids += 1
            continue

        evaluators_with_ids += 1
        models_to_use = (
            [model_filter]
            if model_filter and model_filter in info["oof_probs"]
            else info["models"]
        )

        eval_data = {"_sample_id": sample_ids}
        for mn in models_to_use:
            probs = info["oof_probs"].get(mn)
            if probs is not None and len(probs) == len(sample_ids):
                eval_data[f"{eval_name}_{mn}"] = probs
            elif probs is not None:
                log.warning(
                    "stacking_length_mismatch",
                    evaluator=eval_name,
                    model=mn,
                    n_probs=len(probs),
                    n_ids=len(sample_ids),
                )

        # Also carry the labels for this evaluator
        eval_data["_oof_labels"] = info["oof_labels"]

        if (
            len(eval_data) > 2
        ):  # Has at least one model column beyond _sample_id and _oof_labels
            frames.append(pd.DataFrame(eval_data))

    if not frames:
        raise ValueError(
            f"Cannot build stacking matrix: {evaluators_without_ids} evaluators "
            f"missing oof_sample_ids, {evaluators_with_ids} had IDs but no probs"
        )

    # Outer-join on sample ID to handle evaluators with different sample sets
    stacking = frames[0]
    for df in frames[1:]:
        stacking = pd.merge(
            stacking, df, on="_sample_id", how="outer", suffixes=("", "_dup")
        )
        # Resolve duplicate _oof_labels columns (keep first)
        dup_cols = [c for c in stacking.columns if c.endswith("_dup")]
        if dup_cols:
            stacking = stacking.drop(columns=dup_cols)

    # Extract labels (should be consistent across evaluators)
    y = stacking["_oof_labels"].values.astype(int)
    sample_ids = stacking["_sample_id"].tolist()

    # Drop internal columns
    feature_cols = [
        c for c in stacking.columns if c not in ("_sample_id", "_oof_labels")
    ]
    stacking_features = stacking[feature_cols].copy()

    n_nans = stacking_features.isna().sum().sum()
    log.info(
        "stacking_matrix_built",
        n_samples=len(stacking_features),
        n_features=len(feature_cols),
        n_evaluators=evaluators_with_ids,
        n_nans=int(n_nans),
        feature_cols=feature_cols[:10],
        sample_ids_head=sample_ids[:3],
    )

    return stacking_features, y


def _select_multimodal_features(
    df: pd.DataFrame,
    y: np.ndarray,
    *,
    max_nan_frac: float = 0.80,
    top_percentile: float = 10.0,
    strategy: str = "mi",
    random_state: int = 42,
) -> tuple[pd.DataFrame, list[str]]:
    """Feature selection pipeline for multimodal evaluation.

    Steps:
        1. Drop columns with > ``max_nan_frac`` NaN
        2. Median-impute remaining NaNs
        3. Drop zero-variance columns
        4. Rank by Boruta-SHAP (if ``strategy="boruta_shap"``) or
           mutual information (default/fallback) → keep top percentile.
           If Boruta-SHAP confirms >500 features, MI reduces within
           the confirmed set to ``top_percentile``%.

    Args:
        df: Feature DataFrame (numeric columns only).
        y: Binary label array (same length as ``df``).
        max_nan_frac: Maximum fraction of NaN allowed per column.
        top_percentile: Percentage of features to retain after MI ranking.
        strategy: ``"mi"`` (default) or ``"boruta_shap"``.

    Returns:
        (selected_df, selected_feature_names)
    """
    from sklearn.feature_selection import mutual_info_classif

    n_initial = len(df.columns)

    # Step 1: Drop columns with too many NaNs
    nan_fracs = df.isna().mean()
    high_nan = nan_fracs[nan_fracs > max_nan_frac].index.tolist()
    if high_nan:
        df = df.drop(columns=high_nan)
        log.info(
            "multimodal_feature_nan_drop",
            n_dropped=len(high_nan),
            max_nan_frac=max_nan_frac,
            examples=high_nan[:5],
        )

    # Step 2: Impute remaining NaNs using shared strategy
    n_nans = int(df.isna().sum().sum())
    if n_nans > 0:
        from kreview.selection import _impute

        df = _impute(df, "median")
        log.info("multimodal_feature_imputed", n_nans=n_nans, strategy="median")

    # Step 3: Drop zero-variance
    variances = df.var()
    zero_var = variances[variances <= 0].index.tolist()
    if zero_var:
        df = df.drop(columns=zero_var)
        log.info("multimodal_feature_zerovar_drop", n_dropped=len(zero_var))

    if df.empty or len(df.columns) == 0:
        log.error("multimodal_no_features_after_selection")
        return df, []

    # Step 4: Feature ranking
    if strategy == "boruta_shap" and len(df.columns) > 1:
        # BorutaShap (Ekeany v1.0.17) internally imports scipy.stats.binom_test
        # at module level. binom_test was deprecated in scipy 1.7 and removed
        # in scipy ≥1.12. Patch with the official replacement binomtest(),
        # wrapping .pvalue to match the old return type (bare float).
        import scipy
        import scipy.stats as _ss

        if not hasattr(_ss, "binom_test"):
            _ss.binom_test = lambda x, n, p, alternative="two-sided": (
                _ss.binomtest(x, n, p, alternative).pvalue
            )
            log.info("scipy_binom_test_shimmed", scipy_version=scipy.__version__)

        from BorutaShap import BorutaShap
        from xgboost import XGBClassifier

        log.info("boruta_shap_start", n_features=len(df.columns))

        selector = BorutaShap(
            model=XGBClassifier(
                n_estimators=100,
                max_depth=5,
                use_label_encoder=False,
                eval_metric="logloss",
                random_state=random_state,
                verbosity=0,
            ),
            importance_measure="shap",
            classification=True,
        )
        y_series = pd.Series(y, index=df.index, name="target")
        selector.fit(
            X=df,
            y=y_series,
            n_trials=50,
            sample=False,
            verbose=False,
            random_state=random_state,
        )
        subset = selector.Subset()

        if not subset.empty:
            confirmed = list(subset.columns)
            log.info(
                "boruta_shap_complete",
                n_confirmed=len(confirmed),
                top_5=confirmed[:5],
            )

            # If Boruta confirms >500 features, apply MI reduction
            if len(confirmed) > 500:
                n_keep = max(1, int(len(confirmed) * (top_percentile / 100.0)))
                mi_scores = mutual_info_classif(
                    df[confirmed].values, y, random_state=random_state
                )
                mi_ranked = sorted(
                    zip(confirmed, mi_scores),
                    key=lambda x: x[1],
                    reverse=True,
                )
                confirmed = [name for name, _ in mi_ranked[:n_keep]]
                log.info(
                    "boruta_shap_mi_reduction",
                    n_boruta=len(subset.columns),
                    n_after_mi=len(confirmed),
                    top_percentile=top_percentile,
                    top_5=[(n, f"{s:.3f}") for n, s in mi_ranked[:5]],
                )

            df = df[confirmed]
            return df, confirmed
        else:
            log.warning(
                "boruta_shap_no_features",
                fallback="mi_top_percentile",
                n_features=len(df.columns),
            )
            # Fall through to MI ranking below

    # Step 5: Mutual information ranking (also fallback for boruta_shap)
    n_keep = max(1, int(len(df.columns) * (top_percentile / 100.0)))
    if len(df.columns) > n_keep:
        mi_scores = mutual_info_classif(df.values, y, random_state=random_state)
        mi_ranked = sorted(zip(df.columns, mi_scores), key=lambda x: x[1], reverse=True)
        selected = [name for name, _ in mi_ranked[:n_keep]]
        df = df[selected]
        log.info(
            "multimodal_feature_mi_selected",
            n_before=len(mi_ranked),
            n_after=len(selected),
            top_percentile=top_percentile,
            top_5=[(name, f"{score:.3f}") for name, score in mi_ranked[:5]],
        )
    else:
        selected = list(df.columns)

    log.info(
        "multimodal_feature_selection_complete",
        n_initial=n_initial,
        n_final=len(selected),
        strategy=strategy,
    )
    return df, selected


def multimodal_eval(
    results_dir: str | Path,
    super_matrix_path: str | Path | None = None,
    *,
    models: tuple[str, ...] = ("rf", "xgb"),
    gpu_models: tuple[str, ...] = (),
    device: str = "cuda",
    finetune_epochs: int = 50,
    finetune_lr: float = 1e-5,
    n_folds: int = 5,
    top_percentile: float = 10.0,
    random_state: int = 42,
    multimodal_selection: str = "mi",
) -> dict:
    """Cross-evaluator multimodal evaluation.

    Implements three complementary strategies:

    1. **Stacking**: Meta-learner trained on OOF predictions from all
       per-evaluator models.  Each column is one evaluator's OOF
       probability.  This measures how much combining multiple
       fragmentomics signals improves classification.

    2. **Raw features** (optional): If ``super_matrix_path`` is provided,
       trains directly on the fused feature matrix with
       ``multimodal_selection``-based feature selection (MI or Boruta-SHAP).

    3. **Ablation**: Leave-one-evaluator-out analysis on the stacking
       matrix, showing each evaluator's marginal contribution.

    Args:
        results_dir: Directory with ``*_model_results.json`` files.
        super_matrix_path: Optional path to ``super_matrix.parquet``.
        models: CPU model names to use (``rf``, ``xgb``, ``lr``).
        gpu_models: GPU model names (``tabpfn_ft``, ``tabicl_ft``). Empty = CPU only.
        device: PyTorch device string for GPU models.
        finetune_epochs: Epochs for fine-tuned GPU variants (default 50).
        finetune_lr: Learning rate for fine-tuned GPU variants.
        n_folds: Cross-validation folds.
        top_percentile: Top N% features for feature selection.
        random_state: Reproducibility seed.
        multimodal_selection: Feature selection strategy ('mi' or 'boruta_shap').

    Returns:
        A comprehensive results dict with keys for each strategy.
    """
    from kreview.core import LABEL_META_COLS

    # GPU kwargs for _build_model dispatch
    gpu_kwargs = dict(
        device=device,
        finetune_epochs=finetune_epochs,
        finetune_lr=finetune_lr,
        random_state=random_state,
    )
    # Merge CPU + GPU model lists for unified iteration
    all_models = list(models) + list(gpu_models)

    results = {
        "strategy": "multimodal",
        "models_requested": all_models,
        "gpu_models_requested": list(gpu_models),
    }

    # ── Load baselines ──
    log.info("multimodal_eval_start", results_dir=str(results_dir))
    baselines = _load_per_evaluator_baselines(results_dir)
    results["n_evaluators"] = len(baselines)
    results["evaluators"] = sorted(baselines.keys())

    # Capture per-evaluator best AUCs for comparison
    single_aucs = {}
    for eval_name, info in baselines.items():
        if info.get("best_auc") is not None:
            single_aucs[eval_name] = info["best_auc"]
    results["single_evaluator_aucs"] = single_aucs

    best_single_name = max(single_aucs, key=single_aucs.get) if single_aucs else None
    best_single_auc = (
        single_aucs.get(best_single_name, 0.0) if best_single_name else 0.0
    )
    results["best_single_evaluator"] = best_single_name
    results["best_single_auc"] = best_single_auc

    # ── Strategy 1: Stacking ──
    log.info("multimodal_stacking_start")
    try:
        stacking_df, y_stack = _build_stacking_matrix(baselines)

        # Impute NaNs from outer-join mismatches
        n_nans = int(stacking_df.isna().sum().sum())
        if n_nans > 0:
            stacking_df = stacking_df.fillna(0.5)  # Neutral probability
            log.info("stacking_nan_imputed", n_nans=n_nans, fill_value=0.5)

        cv = StratifiedKFold(
            n_splits=min(n_folds, len(y_stack) // 2),
            shuffle=True,
            random_state=random_state,
        )

        stacking_results = {}
        for model_name in all_models:
            try:
                is_gpu = model_name in _GPU_MODEL_NAMES
                model = _build_model(
                    model_name, random_state, **(gpu_kwargs if is_gpu else {})
                )
                if model is None:
                    log.warning(
                        "stacking_model_skipped",
                        model=model_name,
                        reason="build returned None (import failed?)",
                    )
                    continue
                res, _ = evaluate_model(
                    model,
                    stacking_df.values,
                    y_stack,
                    cv,
                    f"stacking_{model_name}",
                    feature_names=list(stacking_df.columns),
                    random_state=random_state,
                )
                stacking_results.update(res)

                stacking_auc = res.get(f"auc_stacking_{model_name}")
                if stacking_auc is not None and best_single_auc > 0:
                    delta = stacking_auc - best_single_auc
                    stacking_results[f"stacking_{model_name}_vs_best_single"] = delta
                    log.info(
                        "stacking_model_complete",
                        model=model_name,
                        auc=stacking_auc,
                        delta_vs_best_single=delta,
                        is_gpu=is_gpu,
                    )
            except Exception as e:
                log.error("stacking_model_failed", model=model_name, error=str(e))
                stacking_results[f"stacking_{model_name}_error"] = str(e)

        results["stacking"] = stacking_results
        results["stacking_n_features"] = len(stacking_df.columns)
        results["stacking_features"] = list(stacking_df.columns)

    except Exception as e:
        log.error("multimodal_stacking_failed", error=str(e))
        results["stacking_error"] = str(e)

    # ── Strategy 2: Raw features (optional) ──
    if super_matrix_path is not None:
        log.info("multimodal_raw_start", super_matrix=str(super_matrix_path))
        try:
            super_df = pd.read_parquet(super_matrix_path)
            log.info(
                "super_matrix_loaded",
                n_samples=len(super_df),
                n_cols=len(super_df.columns),
            )

            # Extract labels
            if "label" not in super_df.columns:
                raise ValueError("super_matrix must contain 'label' column")

            # Use shared build_binary_target to filter and encode labels
            # — single source of truth for label → binary mapping.
            from kreview.selection import build_binary_target

            super_df, y_raw = build_binary_target(super_df, "label")

            # Select numeric feature columns (exclude metadata)
            feature_cols = [
                c
                for c in super_df.select_dtypes(include=np.number).columns
                if c not in LABEL_META_COLS
            ]

            if not feature_cols:
                raise ValueError("No numeric feature columns in super_matrix")

            X_raw = super_df[feature_cols]
            X_selected, selected_names = _select_multimodal_features(
                X_raw,
                y_raw,
                top_percentile=top_percentile,
                strategy=multimodal_selection,
                random_state=random_state,
            )

            if X_selected.empty:
                raise ValueError("No features survived selection")

            cv_raw = StratifiedKFold(
                n_splits=min(n_folds, len(y_raw) // 2),
                shuffle=True,
                random_state=random_state,
            )

            raw_results = {}
            for model_name in all_models:
                try:
                    is_gpu = model_name in _GPU_MODEL_NAMES
                    model = _build_model(
                        model_name, random_state, **(gpu_kwargs if is_gpu else {})
                    )
                    if model is None:
                        log.warning(
                            "raw_model_skipped",
                            model=model_name,
                            reason="build returned None (import failed?)",
                        )
                        continue
                    res, _ = evaluate_model(
                        model,
                        X_selected.values,
                        y_raw,
                        cv_raw,
                        f"raw_{model_name}",
                        feature_names=selected_names,
                        random_state=random_state,
                    )
                    raw_results.update(res)

                    raw_auc = res.get(f"auc_raw_{model_name}")
                    if raw_auc is not None and best_single_auc > 0:
                        raw_results[f"raw_{model_name}_vs_best_single"] = (
                            raw_auc - best_single_auc
                        )
                except Exception as e:
                    log.error("raw_model_failed", model=model_name, error=str(e))
                    raw_results[f"raw_{model_name}_error"] = str(e)

            results["raw_features"] = raw_results
            results["raw_n_features_selected"] = len(selected_names)
            results["raw_features_selected"] = selected_names[:20]
            results["raw_features_selection_method"] = multimodal_selection

        except Exception as e:
            log.error("multimodal_raw_failed", error=str(e))
            results["raw_features_error"] = str(e)

    # ── Strategy 3: Ablation (leave-one-evaluator-out) ──
    if "stacking" in results and "stacking_error" not in results:
        log.info("multimodal_ablation_start")
        try:
            ablation = {}
            # Use the best stacking model for ablation
            best_stack_model = None
            best_stack_auc = 0.0
            for mn in all_models:
                auc_val = results["stacking"].get(f"auc_stacking_{mn}", 0.0)
                if auc_val and auc_val > best_stack_auc:
                    best_stack_auc = auc_val
                    best_stack_model = mn

            if best_stack_model is not None:
                # Drop each evaluator's columns and retrain
                for eval_name in sorted(baselines.keys()):
                    cols_to_drop = [
                        c for c in stacking_df.columns if c.startswith(f"{eval_name}_")
                    ]
                    if not cols_to_drop:
                        continue

                    X_ablated = stacking_df.drop(columns=cols_to_drop)
                    if X_ablated.empty:
                        continue

                    try:
                        model = _build_model(best_stack_model, random_state)
                        res, _ = evaluate_model(
                            model,
                            X_ablated.values,
                            y_stack,
                            cv,
                            f"ablation_{eval_name}",
                            feature_names=list(X_ablated.columns),
                            random_state=random_state,
                        )
                        ablated_auc = res.get(f"auc_ablation_{eval_name}", 0.0)
                        delta = best_stack_auc - (ablated_auc or 0.0)
                        ablation[eval_name] = {
                            "auc_without": ablated_auc,
                            "delta": delta,
                            "n_cols_removed": len(cols_to_drop),
                        }
                        log.info(
                            "ablation_evaluator",
                            evaluator=eval_name,
                            auc_without=ablated_auc,
                            delta=delta,
                        )
                    except Exception as e:
                        log.error(
                            "ablation_evaluator_failed",
                            evaluator=eval_name,
                            error=str(e),
                        )
                        ablation[eval_name] = {"error": str(e)}

                # Sort by delta (most important first)
                ablation_sorted = dict(
                    sorted(
                        ablation.items(),
                        key=lambda x: x[1].get("delta", 0.0),
                        reverse=True,
                    )
                )
                results["ablation"] = ablation_sorted
                results["ablation_model"] = best_stack_model
                results["ablation_baseline_auc"] = best_stack_auc

        except Exception as e:
            log.error("multimodal_ablation_failed", error=str(e))
            results["ablation_error"] = str(e)

    log.info(
        "multimodal_eval_complete",
        strategies=["stacking"]
        + (["raw_features"] if "raw_features" in results else [])
        + (["ablation"] if "ablation" in results else []),
        n_evaluators=len(baselines),
    )

    return results


# ── Decomposed Multimodal Pipeline ────────────────────────────────────────────
#
# These functions implement the same logic as `multimodal_eval()` but as
# independent stages that can be run in parallel or sequentially.  The
# monolithic `multimodal_eval()` above is preserved for backward compat
# and for local runs that don't need NF-level scattering.
#
# Pipeline:  prep → single × N → ablation → merge
#


def multimodal_prep(
    results_dir: str | Path,
    super_matrix_path: str | Path | None = None,
    *,
    multimodal_selection: str = "mi",
    top_percentile: float = 10.0,
    random_state: int = 42,
    output_dir: str | Path = ".",
) -> dict:
    """Stage 1: Build stacking + optional raw-feature matrices.

    Loads per-evaluator model results, builds a stacking matrix from OOF
    probabilities, optionally loads and selects from the super matrix,
    and writes outputs as parquet files + a metadata JSON.

    Args:
        results_dir: Directory with ``*_model_results.json`` files.
        super_matrix_path: Optional path to ``super_matrix.parquet``.
        multimodal_selection: Feature selection strategy for raw features
            (``"mi"`` or ``"boruta_shap"``).
        top_percentile: Top N% features to retain after MI ranking.
        random_state: Random seed.
        output_dir: Directory to write output files.

    Returns:
        Metadata dict with keys: ``stacking_columns``, ``n_evaluators``,
        ``evaluators``, ``single_evaluator_aucs``, ``best_single_evaluator``,
        ``best_single_auc``, ``stacking_shape``, ``raw_shape`` (optional).

    Raises:
        FileNotFoundError: If ``results_dir`` does not exist.
        ValueError: If no valid evaluator baselines found, or stacking
            matrix has 0 columns.
    """
    from kreview.core import LABEL_META_COLS

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    log.info(
        "multimodal_prep_start",
        results_dir=str(results_dir),
        super_matrix=str(super_matrix_path) if super_matrix_path else None,
        multimodal_selection=multimodal_selection,
        top_percentile=top_percentile,
    )

    # ── Load baselines ──
    baselines = _load_per_evaluator_baselines(results_dir)

    # ── Per-evaluator AUC summary ──
    single_aucs = {}
    for eval_name, info in baselines.items():
        if info.get("best_auc") is not None:
            single_aucs[eval_name] = info["best_auc"]

    best_single_name = max(single_aucs, key=single_aucs.get) if single_aucs else None
    best_single_auc = (
        single_aucs.get(best_single_name, 0.0) if best_single_name else 0.0
    )

    # ── Build stacking matrix ──
    stacking_df, y_stack = _build_stacking_matrix(baselines)

    # Impute NaNs from outer-join mismatches
    n_nans = int(stacking_df.isna().sum().sum())
    if n_nans > 0:
        stacking_df = stacking_df.fillna(0.5)
        log.info("stacking_nan_imputed", n_nans=n_nans, fill_value=0.5)

    if stacking_df.empty or len(stacking_df.columns) == 0:
        raise ValueError(
            "Stacking matrix has 0 columns after building — "
            "no evaluators have OOF probabilities"
        )

    # Save stacking matrix (features + labels)
    stacking_out = stacking_df.copy()
    stacking_out["_label"] = y_stack
    stacking_path = output_dir / "stacking_matrix.parquet"
    stacking_out.to_parquet(stacking_path, index=False)
    log.info(
        "stacking_matrix_saved",
        path=str(stacking_path),
        shape=stacking_df.shape,
        columns_sample=list(stacking_df.columns[:10]),
    )

    # ── Build raw features matrix (optional) ──
    raw_shape = None
    if super_matrix_path is not None:
        super_matrix_path = Path(super_matrix_path)
        if not super_matrix_path.exists():
            raise FileNotFoundError(f"Super matrix not found: {super_matrix_path}")

        super_df = pd.read_parquet(super_matrix_path)
        log.info(
            "super_matrix_loaded",
            n_samples=len(super_df),
            n_cols=len(super_df.columns),
        )

        if "label" not in super_df.columns:
            raise ValueError("super_matrix must contain 'label' column")

        from kreview.selection import build_binary_target

        super_df, y_raw = build_binary_target(super_df, "label")

        feature_cols = [
            c
            for c in super_df.select_dtypes(include=np.number).columns
            if c not in LABEL_META_COLS
        ]
        if not feature_cols:
            raise ValueError("No numeric feature columns in super_matrix")

        X_raw = super_df[feature_cols]
        X_selected, selected_names = _select_multimodal_features(
            X_raw,
            y_raw,
            top_percentile=top_percentile,
            strategy=multimodal_selection,
            random_state=random_state,
        )

        if X_selected.empty:
            raise ValueError("No features survived selection")

        raw_out = X_selected.copy()
        raw_out["_label"] = y_raw
        raw_path = output_dir / "raw_features_matrix.parquet"
        raw_out.to_parquet(raw_path, index=False)
        raw_shape = X_selected.shape
        log.info(
            "raw_features_matrix_saved",
            path=str(raw_path),
            shape=raw_shape,
            selection=multimodal_selection,
            n_selected=len(selected_names),
        )

    # ── Build metadata ──
    import json as _json_mod

    metadata = {
        "stacking_columns": list(stacking_df.columns),
        "stacking_shape": list(stacking_df.shape),
        "n_evaluators": len(baselines),
        "evaluators": sorted(baselines.keys()),
        "single_evaluator_aucs": single_aucs,
        "best_single_evaluator": best_single_name,
        "best_single_auc": best_single_auc,
        "multimodal_selection": multimodal_selection,
        "top_percentile": top_percentile,
        "has_raw_features": raw_shape is not None,
        "raw_shape": list(raw_shape) if raw_shape is not None else None,
    }

    meta_path = output_dir / "prep_metadata.json"
    with open(meta_path, "w") as f:
        _json_mod.dump(metadata, f, indent=2, default=str)
    log.info("multimodal_prep_complete", metadata_path=str(meta_path))

    return metadata


def multimodal_single(
    stacking_matrix_path: str | Path,
    model_name: str,
    *,
    raw_features_path: str | Path | None = None,
    n_folds: int = 5,
    random_state: int = 42,
    device: str = "cuda",
    finetune_epochs: int = 50,
    finetune_lr: float = 1e-5,
    best_single_auc: float = 0.0,
    output_dir: str | Path = ".",
) -> dict:
    """Stage 2: Train a single model on the stacking (+ optional raw) matrix.

    Reads the stacking matrix parquet, trains the requested model via
    :func:`evaluate_model`, and writes a per-model result JSON.

    Args:
        stacking_matrix_path: Path to ``stacking_matrix.parquet`` from prep.
        model_name: Model to train (e.g. ``"rf"``, ``"xgb"``, ``"tabpfn_ft"``).
        raw_features_path: Optional path to ``raw_features_matrix.parquet``.
        n_folds: Cross-validation folds.
        random_state: Random seed.
        device: PyTorch device for GPU models.
        finetune_epochs: Fine-tuning epochs for ``_ft`` models.
        finetune_lr: Fine-tuning learning rate for ``_ft`` models.
        best_single_auc: Best single-evaluator AUC (for delta computation).
        output_dir: Directory to write output JSON.

    Returns:
        Results dict with stacking and optional raw-feature metrics.

    Raises:
        FileNotFoundError: If stacking_matrix_path doesn't exist.
        ValueError: If model build fails.
    """
    import json as _json_mod

    stacking_matrix_path = Path(stacking_matrix_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not stacking_matrix_path.exists():
        raise FileNotFoundError(f"Stacking matrix not found: {stacking_matrix_path}")

    log.info(
        "multimodal_single_start",
        model=model_name,
        stacking_path=str(stacking_matrix_path),
    )

    # Load stacking matrix
    stacking_full = pd.read_parquet(stacking_matrix_path)
    y_stack = stacking_full["_label"].values.astype(int)
    stacking_df = stacking_full.drop(columns=["_label"])

    # Build GPU kwargs
    is_gpu = model_name in _GPU_MODEL_NAMES
    gpu_kwargs = dict(
        device=device,
        finetune_epochs=finetune_epochs,
        finetune_lr=finetune_lr,
        random_state=random_state,
    )

    # Build model
    model = _build_model(model_name, random_state, **(gpu_kwargs if is_gpu else {}))
    if model is None:
        raise ValueError(
            f"Failed to build model '{model_name}' — "
            f"check that required packages are installed"
        )

    cv = StratifiedKFold(
        n_splits=min(n_folds, len(y_stack) // 2),
        shuffle=True,
        random_state=random_state,
    )

    results = {}

    # ── Stacking evaluation ──
    res, _ = evaluate_model(
        model,
        stacking_df.values,
        y_stack,
        cv,
        f"stacking_{model_name}",
        feature_names=list(stacking_df.columns),
        random_state=random_state,
    )
    results.update(res)

    stacking_auc = res.get(f"auc_stacking_{model_name}")
    if stacking_auc is not None and best_single_auc > 0:
        results[f"stacking_{model_name}_vs_best_single"] = (
            stacking_auc - best_single_auc
        )
    log.info(
        "multimodal_single_stacking_done",
        model=model_name,
        auc=stacking_auc,
    )

    # ── Raw features evaluation (optional) ──
    if raw_features_path is not None:
        raw_features_path = Path(raw_features_path)
        if raw_features_path.exists():
            raw_full = pd.read_parquet(raw_features_path)
            y_raw = raw_full["_label"].values.astype(int)
            X_raw = raw_full.drop(columns=["_label"])

            model_raw = _build_model(
                model_name, random_state, **(gpu_kwargs if is_gpu else {})
            )
            if model_raw is not None:
                cv_raw = StratifiedKFold(
                    n_splits=min(n_folds, len(y_raw) // 2),
                    shuffle=True,
                    random_state=random_state,
                )
                raw_res, _ = evaluate_model(
                    model_raw,
                    X_raw.values,
                    y_raw,
                    cv_raw,
                    f"raw_{model_name}",
                    feature_names=list(X_raw.columns),
                    random_state=random_state,
                )
                results.update(raw_res)

                raw_auc = raw_res.get(f"auc_raw_{model_name}")
                if raw_auc is not None and best_single_auc > 0:
                    results[f"raw_{model_name}_vs_best_single"] = (
                        raw_auc - best_single_auc
                    )
                log.info(
                    "multimodal_single_raw_done",
                    model=model_name,
                    auc=raw_auc,
                )
        else:
            log.warning(
                "raw_features_not_found",
                path=str(raw_features_path),
                hint="Skipping raw-feature evaluation for this model",
            )

    # Save per-model JSON
    out_path = output_dir / f"stacking_{model_name}_results.json"
    with open(out_path, "w") as f:
        _json_mod.dump(results, f, indent=2, default=str)
    log.info("multimodal_single_saved", path=str(out_path), model=model_name)

    return results


def multimodal_ablation(
    stacking_matrix_path: str | Path,
    stacking_results_dir: str | Path,
    *,
    n_folds: int = 5,
    random_state: int = 42,
    output_dir: str | Path = ".",
) -> dict:
    """Stage 3: Leave-one-evaluator-out ablation study.

    Finds the best stacking model from partial result JSONs, then for
    each evaluator drops its columns and retrains to measure marginal
    contribution.

    Args:
        stacking_matrix_path: Path to ``stacking_matrix.parquet``.
        stacking_results_dir: Directory with ``stacking_*_results.json``
            files from :func:`multimodal_single`.
        n_folds: Cross-validation folds.
        random_state: Random seed.
        output_dir: Directory to write output JSON.

    Returns:
        Ablation results dict with per-evaluator deltas.

    Raises:
        FileNotFoundError: If inputs don't exist.
        ValueError: If no stacking results found.
    """
    import json as _json_mod

    stacking_matrix_path = Path(stacking_matrix_path)
    stacking_results_dir = Path(stacking_results_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not stacking_matrix_path.exists():
        raise FileNotFoundError(f"Stacking matrix not found: {stacking_matrix_path}")

    log.info(
        "multimodal_ablation_start",
        stacking_path=str(stacking_matrix_path),
        results_dir=str(stacking_results_dir),
    )

    # Load stacking matrix
    stacking_full = pd.read_parquet(stacking_matrix_path)
    y_stack = stacking_full["_label"].values.astype(int)
    stacking_df = stacking_full.drop(columns=["_label"])

    # Find best stacking model from partial results
    best_stack_model = None
    best_stack_auc = 0.0

    for json_path in sorted(stacking_results_dir.glob("stacking_*_results.json")):
        try:
            with open(json_path) as f:
                partial = _json_mod.load(f)
            for k, v in partial.items():
                if (
                    k.startswith("auc_stacking_")
                    and not k.endswith(("_ci_lower", "_ci_upper"))
                    and isinstance(v, (int, float))
                    and v > best_stack_auc
                ):
                    best_stack_auc = v
                    best_stack_model = k.replace("auc_stacking_", "")
        except (_json_mod.JSONDecodeError, OSError) as exc:
            log.warning(
                "ablation_json_read_failed",
                path=str(json_path),
                error=str(exc),
            )

    if best_stack_model is None:
        raise ValueError(
            f"No valid stacking results found in {stacking_results_dir}. "
            f"Run `kreview eval multimodal single` first."
        )

    log.info(
        "ablation_best_model",
        model=best_stack_model,
        auc=best_stack_auc,
    )

    # ── Leave-one-evaluator-out ──
    cv = StratifiedKFold(
        n_splits=min(n_folds, len(y_stack) // 2),
        shuffle=True,
        random_state=random_state,
    )

    # Discover unique evaluator prefixes from column names
    evaluator_names = sorted(
        {col.rsplit("_", 1)[0] for col in stacking_df.columns if "_" in col}
    )

    ablation = {}
    for eval_name in evaluator_names:
        cols_to_drop = [c for c in stacking_df.columns if c.startswith(f"{eval_name}_")]
        if not cols_to_drop:
            continue

        X_ablated = stacking_df.drop(columns=cols_to_drop)
        if X_ablated.empty:
            continue

        try:
            model = _build_model(best_stack_model, random_state)
            res, _ = evaluate_model(
                model,
                X_ablated.values,
                y_stack,
                cv,
                f"ablation_{eval_name}",
                feature_names=list(X_ablated.columns),
                random_state=random_state,
            )
            ablated_auc = res.get(f"auc_ablation_{eval_name}", 0.0)
            delta = best_stack_auc - (ablated_auc or 0.0)
            ablation[eval_name] = {
                "auc_without": ablated_auc,
                "delta": delta,
                "n_cols_removed": len(cols_to_drop),
            }
            log.info(
                "ablation_evaluator",
                evaluator=eval_name,
                auc_without=ablated_auc,
                delta=delta,
                n_cols_removed=len(cols_to_drop),
            )
        except Exception as e:
            log.error(
                "ablation_evaluator_failed",
                evaluator=eval_name,
                error=str(e),
            )
            ablation[eval_name] = {"error": str(e)}

    # Sort by delta (most important first)
    ablation_sorted = dict(
        sorted(
            ablation.items(),
            key=lambda x: x[1].get("delta", 0.0),
            reverse=True,
        )
    )

    results = {
        "ablation": ablation_sorted,
        "ablation_model": best_stack_model,
        "ablation_baseline_auc": best_stack_auc,
    }

    out_path = output_dir / "ablation_results.json"
    with open(out_path, "w") as f:
        _json_mod.dump(results, f, indent=2, default=str)
    log.info("multimodal_ablation_complete", path=str(out_path))

    return results


def multimodal_merge(
    stacking_results_dir: str | Path,
    prep_metadata_path: str | Path,
    *,
    ablation_path: str | Path | None = None,
    output_dir: str | Path = ".",
) -> dict:
    """Stage 4: Merge partial JSONs into unified ``multimodal_results.json``.

    Combines the prep metadata, per-model stacking results, and ablation
    results into a single output JSON matching the schema produced by the
    monolithic :func:`multimodal_eval`.

    Args:
        stacking_results_dir: Directory with ``stacking_*_results.json``
            files from :func:`multimodal_single`.
        prep_metadata_path: Path to ``prep_metadata.json`` from prep.
        ablation_path: Optional path to ``ablation_results.json``.
        output_dir: Directory to write ``multimodal_results.json``.

    Returns:
        Unified results dict matching :func:`multimodal_eval` schema.

    Raises:
        FileNotFoundError: If prep_metadata_path doesn't exist.
    """
    import json as _json_mod

    prep_metadata_path = Path(prep_metadata_path)
    stacking_results_dir = Path(stacking_results_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not prep_metadata_path.exists():
        raise FileNotFoundError(f"Prep metadata not found: {prep_metadata_path}")

    log.info(
        "multimodal_merge_start",
        metadata_path=str(prep_metadata_path),
        results_dir=str(stacking_results_dir),
    )

    with open(prep_metadata_path) as f:
        metadata = _json_mod.load(f)

    # Build unified results dict
    results = {
        "strategy": "multimodal",
        "n_evaluators": metadata["n_evaluators"],
        "evaluators": metadata["evaluators"],
        "single_evaluator_aucs": metadata["single_evaluator_aucs"],
        "best_single_evaluator": metadata["best_single_evaluator"],
        "best_single_auc": metadata["best_single_auc"],
        "stacking_n_features": metadata["stacking_shape"][1],
        "stacking_features": metadata["stacking_columns"],
    }

    # Merge all stacking result JSONs
    stacking_results = {}
    raw_results = {}
    n_merged = 0

    for json_path in sorted(stacking_results_dir.glob("stacking_*_results.json")):
        try:
            with open(json_path) as f:
                partial = _json_mod.load(f)
            for k, v in partial.items():
                if k.startswith("auc_stacking_") or k.startswith("stacking_"):
                    stacking_results[k] = v
                elif k.startswith("auc_raw_") or k.startswith("raw_"):
                    raw_results[k] = v
            n_merged += 1
        except (_json_mod.JSONDecodeError, OSError) as exc:
            log.warning(
                "merge_json_read_failed",
                path=str(json_path),
                error=str(exc),
            )

    if n_merged == 0:
        log.warning(
            "merge_no_stacking_results",
            dir=str(stacking_results_dir),
            hint="No stacking_*_results.json files found",
        )

    results["stacking"] = stacking_results

    if raw_results:
        results["raw_features"] = raw_results
        if metadata.get("has_raw_features"):
            results["raw_n_features_selected"] = metadata["raw_shape"][1]
            results["raw_features_selection_method"] = metadata["multimodal_selection"]

    # Merge ablation results
    if ablation_path is not None:
        ablation_path = Path(ablation_path)
        if ablation_path.exists():
            try:
                with open(ablation_path) as f:
                    ablation_data = _json_mod.load(f)
                results.update(ablation_data)
            except (_json_mod.JSONDecodeError, OSError) as exc:
                log.warning(
                    "merge_ablation_read_failed",
                    path=str(ablation_path),
                    error=str(exc),
                )
        else:
            log.warning(
                "merge_ablation_not_found",
                path=str(ablation_path),
                hint="Ablation results will be missing from merged output",
            )

    # Save unified output
    out_path = output_dir / "multimodal_results.json"
    with open(out_path, "w") as f:
        _json_mod.dump(results, f, indent=2, default=str)

    log.info(
        "multimodal_merge_complete",
        path=str(out_path),
        n_stacking_files=n_merged,
        has_ablation="ablation" in results,
        has_raw="raw_features" in results,
    )

    return results


def _build_model(model_name: str, random_state: int, **kwargs):
    """Instantiate a model by name for multimodal evaluation.

    Centralises model construction so that ``multimodal_eval`` does not
    duplicate sklearn import/init logic for every strategy × model
    combination.

    Args:
        model_name: One of ``lr``, ``rf``, ``xgb``, ``tabpfn``,
            ``tabpfn_ft``, ``tabicl``, ``tabicl_ft``.
        random_state: Seed for reproducibility.
        **kwargs: GPU model params (``device``, ``finetune_epochs``,
            ``finetune_lr``). Ignored for CPU models.

    Returns:
        An sklearn-compatible estimator, or None if GPU model import fails.

    Raises:
        ValueError: If ``model_name`` is not recognised.
    """
    # _GPU_MODEL_NAMES used from module-level constant

    if model_name == "lr":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                ("lr", LogisticRegression(max_iter=1000, random_state=random_state)),
            ]
        )
    elif model_name == "rf":
        return RandomForestClassifier(
            n_estimators=200, max_depth=6, random_state=random_state, n_jobs=-1
        )
    elif model_name == "xgb":
        try:
            from xgboost import XGBClassifier

            return XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.1,
                random_state=random_state,
                use_label_encoder=False,
                eval_metric="logloss",
                verbosity=0,
                n_jobs=-1,
            )
        except ImportError:
            log.error("xgb_import_failed", hint="pip install xgboost")
            raise
    elif model_name in _GPU_MODEL_NAMES:
        # Delegate to GPU model factory; returns None on ImportError (logged there)
        return _build_gpu_model(
            model_name,
            device=kwargs.get("device", "cuda"),
            finetune_epochs=kwargs.get("finetune_epochs", 50),
            finetune_lr=kwargs.get("finetune_lr", 1e-5),
            random_state=random_state,
        )
    else:
        raise ValueError(
            f"Unknown model: {model_name!r}. "
            f"Choose from: lr, rf, xgb, tabpfn, tabpfn_ft, tabicl, tabicl_ft"
        )